# Chi² analysis on ratio D00/D60
Le Chi² est pertinent pour tester les associations entre la sévérité, le sexe, la présence d’anomalies cliniques et des seuils lipidiques. Il permet d’évaluer si ces facteurs influencent la distribution des patients dans les groupes étudiés.  
  
Les variables catégorielles que l'on peut analyser avec un test du Chi² sont celles qui classent les patients en groupes distincts.  
  
On considère : 
- ∗ p <0.05. 
- ∗∗ p <0.01 et 
- ∗∗∗ p <0.001

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
import scipy.stats as stats
from statsmodels.stats.multitest import multipletests

# Tableaux de contingence des 43 patients

In [2]:
# Charger le fichier Excel
file_path = "/Users/loictalignani/research/project/lipidomics/data/raw_data/Patient_data.xlsx"
xls = pd.ExcelFile(file_path)

# Charger les données de l'onglet
patient_data = pd.read_excel(xls, sheet_name="patient_data")
patient_data

,Patient code,Age,Sex,severity,PBMC,Date of fever,Day of Fever
0,BS-064,14,M,severe,1.76,2022-06-18,0
1,JV-035,13,M,severe,11.43,2022-07-05,7
2,KT-515,29,M,severe,1.80,2023-08-24,6
3,KT-926,8,M,severe,6.30,2023-11-09,4
4,JV-148,14,M,severe,5.65,2023-10-23,0
5,JV-157,13,M,severe,7.00,2023-11-08,5
6,KT-193,10,M,severe,1.90,2023-05-13,3
7,KT-247,10,F,severe,2.00,2023-07-02,2
8,KT-312,14,M,severe,1.10,2023-07-18,2
9,KT-347,9,M,severe,6.05,2023-07-28,3


## Statistiques de base

In [307]:
# Age moyen des patients
age_mean = patient_data.Age.mean()
age_std = patient_data.Age.std()
print(f'Age moyen des patients : {age_mean:.2f} ± {age_std:.2f}')
print('Age médian des patients : ', patient_data.Age.median())

Age moyen des patients : 11.22 ± 4.85
Age médian des patients :  10.5


In [240]:
# Calculer le nb et pourcentage de valeurs "M" dans la colonne "Sex"
nb_m = (patient_data.Sex.value_counts()['M'])
percentage_m = (patient_data['Sex'].value_counts(
    normalize=True)['M'] * 100).round(2)

print(
    f"Sex (male) (Total): {nb_m} ({percentage_m}%)")

Sex (male) (Total): 30 (69.77%)


In [308]:
# Filtrer les données par sexe
mild_sex = patient_data[patient_data["severity"] == "mild"]["Sex"]
severe_sex = patient_data[patient_data["severity"] == "severe"]["Sex"]

nb_mild_sex = mild_sex.value_counts()['M']
percentage_mild_sex= (mild_sex.value_counts(normalize=True)['M'] * 100).round(2)

nb_severe_sex = severe_sex.value_counts()['M']
percentage_severe_sex = (severe_sex.value_counts(normalize=True)['M'] * 100).round(2)

print(f"Sex (male) (Mild): {nb_mild_sex} ({percentage_mild_sex:.2f}%)")
print(f"Sex (male) (Severe): {nb_severe_sex} ({percentage_severe_sex:.2f}%)")

Sex (male) (Mild): 12 (54.55%)
Sex (male) (Severe): 15 (83.33%)


## Tableau de contingence Sex / Severity

In [309]:
print(pd.crosstab(patient_data["Sex"], patient_data["severity"]))

severity  mild  severe
Sex                   
F           10       3
M           12      15


In [310]:
# Créer un tableau de contingence
contingency_table = pd.crosstab(patient_data["Sex"], patient_data["severity"])

# Effectuer le test du chi-carré
# Effectuer le test du chi-carré
chi2, p, dof, expected = chi2_contingency(contingency_table)

# Afficher les résultats
print(f"Chi-square value: {chi2:.2f}")
print(f"P-value: {p:.4f}")

# Interprétation des résultats
alpha = 0.05
if p < alpha:
    print("Il existe une association significative entre le sexe et la sévérité.")
else:
    print("Il n'existe pas d'association significative entre le sexe et la sévérité.")

Chi-square value: 2.54
P-value: 0.1108
Il n'existe pas d'association significative entre le sexe et la sévérité.


## Tableau de contingence Age / Severity

In [311]:
# Création de deux groupes : Child et Teen/Adult, en tenant compte de l'âge moyen
patient_data["Age_Group"] = np.where(
    patient_data["Age"] < 11, "Child", "Teen/Adult")

In [312]:
# Tableau de contingence Age_Group et severity
print(pd.crosstab(patient_data["Age_Group"], patient_data["severity"]))

severity    mild  severe
Age_Group               
Child         13       7
Teen/Adult     9      11


In [313]:
# Filtrer les données par sévérité
mild_ages = patient_data[patient_data["severity"] == "mild"]["Age"]
severe_ages = patient_data[patient_data["severity"] == "severe"]["Age"]

mild_ages_mean = mild_ages.mean()
mild_ages_std = mild_ages.std()
severe_ages_mean = severe_ages.mean()
severe_ages_std = severe_ages.std()

# Affichage des résultats
print(
    f"Âge moyen (Mild): {mild_ages_mean:.2f} ± {mild_ages_std:.2f}")
print(
    f"Âge moyen (Severe): {severe_ages_mean:.2f} ± {severe_ages_std:.2f}")

Âge moyen (Mild): 9.86 ± 4.09
Âge moyen (Severe): 12.89 ± 5.30


In [314]:
# Créer un tableau de contingence
contingency_table = pd.crosstab(patient_data["Age"], patient_data["severity"])

# Effectuer le test du chi-carré
# Effectuer le test du chi-carré
chi2, p, dof, expected = chi2_contingency(contingency_table)

# Afficher les résultats
print(f"Chi-square value: {chi2:.2f}")
print(f"P-value: {p:.4f}")

# Interprétation des résultats
alpha = 0.05
if p < alpha:
    print("Il existe une association significative entre l'âge et la sévérité.")
else:
    print("Il n'existe pas d'association significative entre l'âge et la sévérité.")

Chi-square value: 13.15
P-value: 0.5147
Il n'existe pas d'association significative entre l'âge et la sévérité.


In [315]:
t_stat, p_value = stats.ttest_ind(mild_ages, severe_ages, equal_var=False)
print(f"T-test entre Mild et Severe: t = {t_stat:.3f}, p = {p_value:.5f}")

T-test entre Mild et Severe: t = -1.986, p = 0.05578


p > 0.05, la différence d’âge n'est pas statistiquement significative entre les groupes severe et mild.

## Analyse de la relation Nb PBMC (en millions) et sévérité
Le test du Chi² est utilisé pour tester l’indépendance entre deux variables catégorielles. Or, ici :  
	•	“Sévérité” est une variable catégorielle (“mild” / “severe”). ✅  
	•	“PBMC” est une variable continue (valeur en millions). ❌

👉 Le test du Chi² n’est pas adapté car il compare la fréquence des catégories dans un tableau de contingence, et non la relation entre une variable numérique et une variable catégorielle.   
  
Nous allons plutôt réaliser un test T de student ou un test non-paramétrique de Mann & Whitney, après avoir testé la normalité de la distribution des données avec le test de Shapiro.

### Test de Shapiro : normalité des données

In [316]:
# Séparer les PBMC en fonction de la sévérité
pbmc_severe = patient_data[patient_data["severity"] == "severe"]["PBMC"]
pbmc_mild = patient_data[patient_data["severity"] == "mild"]["PBMC"]

pbmc_severe_mean = pbmc_severe.mean()
pbmc_severe_std = pbmc_severe.std()
pbmc_mild_mean = pbmc_mild.mean()
pbmc_mild_std = pbmc_mild.std()

# Test de normalité (Shapiro-Wilk)
stat, p_value_norm_severe = stats.shapiro(pbmc_severe)
print(f"stat for severe: {stat:.2f}, p: {p_value_norm_severe:.5f}")
stat, p_value_norm_mild = stats.shapiro(pbmc_mild)
print(f"stat for mild: {stat:.2f}, p: {p_value_norm_mild:.5f}")

stat for severe: 0.83, p: 0.00376
stat for mild: 0.87, p: 0.00813


	•	Hypothèse nulle (H₀) : Les données suivent une distribution normale.
	•	Hypothèse alternative (H₁) : Les données ne suivent pas une distribution normale.
	•	p-value < 0.05 : On rejette H₀ → les données ne sont pas normales.
	•	p-value ≥ 0.05 : On ne rejette pas H₀ → les données peuvent être considérées comme normales.

In [317]:
print(f"pbmc severe (millions): {pbmc_severe_mean:.2f} ± {pbmc_severe_std:.2f}")
print(f"pbmc mild (millions): {pbmc_mild_mean:.2f} ± {pbmc_mild_std:.2f}")

pbmc severe (millions): 4.14 ± 3.33
pbmc mild (millions): 3.94 ± 2.40


### Test T de Student / Test de Mann-Whitney

In [318]:
# Si les deux groupes suivent une loi normale, on utilise le test t de Student
if p_value_norm_severe > 0.05 and p_value_norm_mild > 0.05:
    t_stat, p_value = stats.ttest_ind(pbmc_severe, pbmc_mild, equal_var=False)
    test_type = "Test t de Student"
else:
    # Sinon, on utilise un test non paramétrique (Mann-Whitney)
    t_stat, p_value = stats.mannwhitneyu(
        pbmc_severe, pbmc_mild, alternative='two-sided')
    test_type = "Test de Mann-Whitney"

print(f"{test_type} : Statistique = {t_stat:.3f}, p-value = {p_value:.3f}")

Test de Mann-Whitney : Statistique = 178.000, p-value = 0.596


Pas de différence significative entre les groupes mild et severe pour les PBMC.  
➝ La distribution des PBMC semble similaire dans les deux groupes.  
➝ Rien ne permet de dire que la sévérité influence la quantité de PBMC.  

# D0 - données non normalisées avec D60

In [41]:
# Charger le fichier Excel
file_path = "/Users/loictalignani/research/project/lipidomics/data/lipidsig_datasets/somme_des_espèces_lipides_D0.xlsx"
xls = pd.ExcelFile(file_path)

# Charger les données des onglets
sum_species = pd.read_excel(xls, sheet_name="sum_species")
patient_data = pd.read_excel(xls, sheet_name="patient_data")

In [42]:
sum_species

,Family,BS-064D00,BS-082D00,BS-336D00,BS-346D00,BS-351D00,BS-364D00,BS-377D00,BS-671D00,JV-035D00,...,KT-695D00,KT-705D00,KT-716D00,KT-718D00,KT-723D00,KT-771D00,KT-805D00,KT-880D00,KT-926D00,KT-974D00
0,DG,9.534447,7.62489,4.722339,1.558664,8.467026,3.897469,15.211302,4.680918,60.214102,...,4.419748,17.009163,16.755152,13.173997,2.745942,15.849938,23.269528,10.806676,29.573164,2.65621
1,CAR,0.179159,0.618445,0.883822,0.837799,0.68767,1.147424,0.916518,0.861717,0.329057,...,1.114927,0.344134,1.647444,0.178381,1.085803,0.42719,1.9451,0.0,0.439972,1.027213
2,CE,7.760417,2.880691,2.032174,4.008069,4.651314,5.254078,9.4073,1.54218,0.972785,...,2.998043,2.708503,1.755861,2.922082,3.301828,0.0,8.518607,0.0,0.0,4.970909
3,Cer,5.105788,2.08739,1.360267,3.391432,4.487805,4.652435,2.622631,3.137028,2.996562,...,2.292107,2.778303,2.101445,3.01655,2.139231,4.888509,2.245213,1.73043,1.504139,1.3562
4,Chol,11.286518,6.291542,6.413234,14.619689,6.673736,20.374974,7.680811,5.074571,9.816619,...,3.401695,9.944598,8.74197,8.030799,6.272608,9.023244,5.583694,6.580752,3.479196,2.290283
5,HexCer,4.688535,0.0,0.0,0.0,3.391048,1.800382,0.796528,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,2.237946,0.0,0.0,0.0,0.0
6,LPC,1595.862814,1178.460673,2484.710063,3193.412612,1393.371398,2864.027463,2496.057411,1920.74489,2825.769206,...,1409.941995,1902.790563,4410.176994,2424.034878,2008.677233,2487.394191,4247.985858,1285.414163,2560.284895,856.494209
7,PC,1230.068372,569.089308,994.601819,912.379173,922.631327,872.315089,1293.16717,784.485467,1578.296132,...,523.747302,1104.640678,1112.956729,982.788823,601.696692,1098.566715,1242.601443,944.759301,953.615945,384.049678
8,PE,7.655925,0.0,0.0,0.0,7.782003,14.467283,5.879141,0.0,7.673788,...,4.212518,8.145487,6.093268,0.0,8.997636,13.814243,11.146481,8.501925,0.0,0.0
9,SM,470.149453,197.339077,262.694366,377.081354,305.153037,267.771798,318.375433,287.17671,262.990952,...,244.200976,278.619518,300.04174,230.258755,239.63352,412.261908,379.062052,257.38216,143.147457,160.416971


In [43]:
patient_data

,Patient code,Age,Sex,severity
0,BS-064,14,M,severe
1,JV-035,13,M,severe
2,JV-048,12,M,severe
3,KT-515,29,M,severe
4,KT-926,8,M,severe
5,JV-148,14,M,severe
6,JV-157,13,M,severe
7,KT-193,10,M,severe
8,KT-247,10,F,severe
9,KT-312,14,M,severe


In [44]:
# Extraire les informations des patients à partir des colonnes
# Exclure la colonne "Family" et "severity"
sample_cols = sum_species.columns[1:-1]
patient_info = pd.DataFrame({
    # Retirer le jour (D0, D3...) pour matcher patient_data
    "Sample": [col[:-3] for col in sample_cols],
    "Day": [col[-3:] for col in sample_cols],
    "Column": sample_cols,
    # Dernière ligne contient la sévérité
    "Severity": sum_species.iloc[-1, 1:-1].values
})

In [45]:
patient_info

,Sample,Day,Column,Severity
0,BS-064,D00,BS-064D00,severe
1,BS-082,D00,BS-082D00,mild
2,BS-336,D00,BS-336D00,mild
3,BS-346,D00,BS-346D00,mild
4,BS-351,D00,BS-351D00,mild
5,BS-364,D00,BS-364D00,mild
6,BS-377,D00,BS-377D00,mild
7,BS-671,D00,BS-671D00,mild
8,JV-035,D00,JV-035D00,severe
9,JV-138,D00,JV-138D00,mild


In [46]:
# Fusionner avec patient_data pour récupérer le sexe
patient_data = patient_data.rename(columns={"Patient code": "Sample"})
patient_info = patient_info.merge(patient_data, on="Sample", how="left")

In [47]:
patient_info.set_index("Column", inplace=True)
patient_info

,Sample,Day,Severity,Age,Sex,severity
Column,,,,,,
BS-064D00,BS-064,D00,severe,14,M,severe
BS-082D00,BS-082,D00,mild,18,M,mild
BS-336D00,BS-336,D00,mild,11,F,mild
BS-346D00,BS-346,D00,mild,8,F,mild
BS-351D00,BS-351,D00,mild,7,M,mild
BS-364D00,BS-364,D00,mild,6,M,mild
BS-377D00,BS-377,D00,mild,13,F,mild
BS-671D00,BS-671,D00,mild,8,F,mild
JV-035D00,JV-035,D00,severe,13,M,severe


In [48]:
# Appliquer la transformation log2(x + 0.01) sur les valeurs de lipides
lipid_data = sum_species.iloc[:-1, 1:-1].map(lambda x: np.log2(x + 0.01))
lipid_data

,BS-064D00,BS-082D00,BS-336D00,BS-346D00,BS-351D00,BS-364D00,BS-377D00,BS-671D00,JV-035D00,JV-138D00,...,KT-663D00,KT-695D00,KT-705D00,KT-716D00,KT-718D00,KT-723D00,KT-771D00,KT-805D00,KT-880D00,KT-926D00
0,3.254662,2.932607,2.242554,0.649537,3.083558,1.966234,3.928020,2.229870,5.912269,3.404451,...,0.508182,2.147225,4.089088,4.067394,3.720716,1.462546,3.987315,4.540990,3.435185,4.886704
1,-2.402330,-0.670142,-0.161941,-0.238207,-0.519384,0.210917,-0.110110,-0.198068,-1.560401,-6.643856,...,-1.045158,0.169832,-1.497631,0.728960,-2.408275,0.131989,-1.193668,0.967242,-6.643856,-1.152094
2,2.957992,1.531414,1.030106,2.006502,2.220737,2.396181,3.235314,0.634296,-0.025053,3.318275,...,2.484074,1.588825,1.442812,0.820371,1.551925,1.727628,-6.643856,3.092310,-6.643856,-6.643856
3,2.354957,1.068595,0.454457,1.766142,2.169221,2.221084,1.396506,1.653990,1.588115,2.231918,...,1.227451,1.202955,1.479387,1.078231,1.597674,1.103820,2.292343,1.173264,0.799444,0.598498
4,3.497806,2.655705,2.683300,3.870827,2.740655,4.349434,2.943136,2.346126,3.296695,4.772171,...,3.791213,1.770489,3.315363,3.129608,3.007339,2.651364,3.175244,2.483801,2.720443,1.802895
5,2.232211,-6.643856,-6.643856,-6.643856,1.765979,0.856294,-0.310204,-6.643856,-6.643856,2.246656,...,-0.259251,-6.643856,-6.643856,-6.643856,-6.643856,-6.643856,1.168607,-6.643856,-6.643856,-6.643856
6,10.640130,10.202700,11.278868,11.640888,10.444374,11.483835,11.285441,10.907458,11.464433,11.604441,...,10.941087,10.461430,10.893909,12.106624,11.243201,10.972037,11.280425,12.052567,10.328029,11.322094
7,10.264535,9.152537,9.957990,9.833506,9.849626,9.768722,10.336704,9.615621,10.624161,10.240643,...,9.274100,9.032755,10.109375,10.120195,9.940752,9.232917,10.101420,10.279160,9.883818,9.897280
8,2.938460,-6.643855,-6.643854,-6.643853,2.961994,3.855719,2.558057,-6.643853,2.941818,-6.643854,...,2.955865,2.078103,3.027771,2.609582,-6.643853,3.171149,3.789129,3.479810,3.089485,-6.643855
9,8.877006,7.624606,8.037296,8.558770,8.253436,8.064914,8.314631,8.165845,8.038924,8.759690,...,8.268532,7.931984,8.122204,8.229067,7.847175,7.904746,8.687452,8.566328,8.007824,7.161459


In [49]:
# Déterminer un seuil pour chaque famille de lipides (ex : médiane)
thresholds = lipid_data.median(axis=1)
thresholds

0      3.106749
1     -0.562255
2      1.531414
3      1.227451
4      2.831372
5     -6.643856
6     10.907458
7      9.849626
8      1.835982
9      8.037296
10     5.948523
11     8.626705
12     3.711432
dtype: float64

In [50]:
# Binariser les valeurs des lipides (1 si supérieur au seuil, 0 sinon)
binary_lipid_data = lipid_data.gt(thresholds, axis=0).astype(int)
binary_lipid_data

,BS-064D00,BS-082D00,BS-336D00,BS-346D00,BS-351D00,BS-364D00,BS-377D00,BS-671D00,JV-035D00,JV-138D00,...,KT-663D00,KT-695D00,KT-705D00,KT-716D00,KT-718D00,KT-723D00,KT-771D00,KT-805D00,KT-880D00,KT-926D00
0,1,0,0,0,0,0,1,0,1,1,...,0,0,1,1,1,0,1,1,1,1
1,0,0,1,1,1,1,1,1,0,0,...,0,1,0,1,0,1,0,1,0,0
2,1,0,0,1,1,1,1,0,0,1,...,1,1,0,0,1,1,0,1,0,0
3,1,0,0,1,1,1,1,1,1,1,...,0,0,1,0,1,0,1,0,0,0
4,1,0,0,1,0,1,1,0,1,1,...,1,0,1,1,1,0,1,0,0,0
5,1,0,1,1,1,1,1,1,0,1,...,1,0,0,0,0,0,1,0,1,0
6,0,0,1,1,0,1,1,0,1,1,...,1,0,0,1,1,1,1,1,0,1
7,1,0,1,0,0,0,1,0,1,1,...,0,0,1,1,1,0,1,1,1,1
8,1,0,0,0,1,1,1,0,1,0,...,1,1,1,1,0,1,1,1,1,0
9,1,0,0,1,1,1,1,1,1,1,...,1,0,1,1,0,0,1,1,0,0


## Test du Chi² sur les lipides en fonction de la sévérité

In [51]:
# Effectuer les tests du Chi²
chi2_results = {}

# Chi² entre la sévérité et la présence des familles de lipides
for i, lipid in enumerate(sum_species.iloc[:-1, 0]):
    contingency_table = pd.crosstab(
        binary_lipid_data.iloc[i, :], patient_info["Severity"])
    chi2, p, _, _ = chi2_contingency(contingency_table)
    chi2_results[lipid] = {"Chi2": chi2, "p-value": p}

In [52]:
chi2_results

{'DG': {'Chi2': 5.747979323308268, 'p-value': 0.016507634005564046},
 'CAR': {'Chi2': 4.413768796992484, 'p-value': 0.035649996457949557},
 'CE': {'Chi2': 0.6652725563909783, 'p-value': 0.41470468054923515},
 'Cer': {'Chi2': 0.0, 'p-value': 1.0},
 'Chol': {'Chi2': 0.0, 'p-value': 1.0},
 'HexCer': {'Chi2': 4.413768796992484, 'p-value': 0.035649996457949557},
 'LPC': {'Chi2': 2.126550751879701, 'p-value': 0.14476619644361308},
 'PC': {'Chi2': 3.0795582706766904, 'p-value': 0.07928205591587303},
 'PE': {'Chi2': 0.029934210526315973, 'p-value': 0.862639591019358},
 'SM': {'Chi2': 2.126550751879701, 'p-value': 0.14476619644361308},
 'TG': {'Chi2': 3.0795582706766904, 'p-value': 0.07928205591587303},
 'FA': {'Chi2': 1.2370770676691718, 'p-value': 0.2660351186466848},
 'PI': {'Chi2': 5.747979323308268, 'p-value': 0.016507634005564046}}

DG, CAR, HexCer et PI montrent une association statistiquement significative avec la sévérité de la dengue. Cela suggère que la présence (ou l’absence) de ces lipides pourrait être liée à la gravité de la maladie.

## Test du Chi² sur le sexe en fonction de la sévérité

In [53]:
# Chi² entre le sexe et la sévérité
contingency_sex = pd.crosstab(patient_info["Sex"], patient_info["Severity"])
chi2_sex, p_sex, _, _ = chi2_contingency(contingency_sex)
chi2_results["Sex vs Severity"] = {"Chi2": chi2_sex, "p-value": p_sex}

In [54]:
print(chi2_sex)
print(p_sex)

2.901785714285715
0.0884814827675487


Sex vs Severity a une p-value > 0.05 (0.088), indiquant une faible relation entre le sexe des patients et la sévérité de la maladie. Cela signifie que la sévérité est distribuée ééquitablement selon le sexe.

### Tableau de contingence Sex / Severity

In [58]:
print(pd.crosstab(patient_info["Sex"], patient_info["Severity"]))

Severity  mild  severe
Sex                   
F           10       3
M           11      15


### Résidus standardisés 
Quelle catégorie contribue le plus au Chi²

In [59]:
table = pd.crosstab(patient_info["Sex"], patient_info["Severity"])
chi2, p, dof, expected = chi2_contingency(table)
residuals = (table - expected) / np.sqrt(expected)
print(residuals)

Severity      mild    severe
Sex                         
F         1.133893 -1.224745
M        -0.801784  0.866025


## Test du Chi² sur l'âge en fonction de la sévérité

In [55]:
# Test du Chi² pour l'âge (regroupé en deux catégories)
patient_info["Age_Group"] = np.where(
    patient_info["Age"] < 14, "Child", "Teen/Adult")

# Table de contingence Age vs Sévérité
contingency_table_age = pd.crosstab(
    patient_info["Age_Group"], patient_info["Severity"])
chi2_age, p_age, _, _ = chi2_contingency(contingency_table_age)
chi2_results["Age vs Severity"] = {"Chi2": chi2_age, "p-value": p_age}

# Convertir les résultats en DataFrame
chi2_df = pd.DataFrame.from_dict(chi2_results, orient="index")

In [56]:
chi2_df

,Chi2,p-value
DG,5.747979,0.016508
CAR,4.413769,0.035650
CE,0.665273,0.414705
Cer,0.000000,1.000000
Chol,0.000000,1.000000
HexCer,4.413769,0.035650
LPC,2.126551,0.144766
PC,3.079558,0.079282
PE,0.029934,0.862640
SM,2.126551,0.144766


Age vs Severity a une p-value < 0.05 (0.034), indiquant une forte relation entre l'âge des patients et la sévérité de la maladie. Cela signifie que la sévérité est distribuée différement selon l'âge.

### Tableau de contingence Age / Severity

In [61]:
print(pd.crosstab(patient_info["Age_Group"], patient_info["Severity"]))

Severity    mild  severe
Age_Group               
Child         19      10
Teen/Adult     2       8


### Calcul des résidus standardisés

In [62]:
table = pd.crosstab(patient_info["Age_Group"], patient_info["Severity"])
chi2, p, dof, expected = chi2_contingency(table)
residuals = (table - expected) / np.sqrt(expected)
print(residuals)

Severity        mild    severe
Age_Group                     
Child       0.856511 -0.925138
Teen/Adult -1.458586  1.575453


In [ ]:
# Convertir les résultats en DataFrame et sauvegarder
chi2_df = pd.DataFrame.from_dict(chi2_results, orient="index")


Analyse Chi² terminée. Résultats enregistrés dans chi2_results.csv


In [ ]:
chi2_df.to_csv(
    "/Users/loictalignani/research/project/lipidomics/analysis/univariate_analysis/chi2_results_D0.csv", index=True)

print("Analyse Chi² terminée. Résultats enregistrés dans chi2_results.csv")

## Benjamini-Hochberg correction
Plusieurs tests en parallèle sont effectués, ce qui augmente le risque de faux positifs. La correction BH ajuste les p-values pour tenir compte de ce biais. Elle contrôle le taux de fausses découvertes (FDR) plutôt que le taux d’erreur global. C’est plus adapté que Bonferroni, qui est trop strict. Les p-values corrigées sont souvent plus grandes, ce qui diminue le risque d’interpréter à tort un résultat comme significatif.

In [63]:
from statsmodels.stats.multitest import multipletests

In [64]:
# Correction de Benjamini-Hochberg pour toutes les p-values
_, corrected_p_values, _, _ = multipletests(
    chi2_df["p-value"], method="fdr_bh")

# Ajouter les p-values corrigées au DataFrame
chi2_df["p-value (BH corrected)"] = corrected_p_values

# Afficher les résultats
print(chi2_df)

                     Chi2   p-value  p-value (BH corrected)
DG               5.747979  0.016508                0.106950
CAR              4.413769  0.035650                0.106950
CE               0.665273  0.414705                0.518381
Cer              0.000000  1.000000                1.000000
Chol             0.000000  1.000000                1.000000
HexCer           4.413769  0.035650                0.106950
LPC              2.126551  0.144766                0.217149
PC               3.079558  0.079282                0.165903
PE               0.029934  0.862640                0.995353
SM               2.126551  0.144766                0.217149
TG               3.079558  0.079282                0.165903
FA               1.237077  0.266035                0.362775
PI               5.747979  0.016508                0.106950
Sex vs Severity  2.901786  0.088481                0.165903
Age vs Severity  4.502771  0.033840                0.106950


In [67]:
chi2_df.to_csv(
    "/Users/loictalignani/research/project/lipidomics/analysis/univariate_analysis/chi2_results_bh_corrected_D0.csv", index=True)

print("Analyse Chi² terminée. Résultats enregistrés dans chi2_results_bh_corrected_D0.csv")

Analyse Chi² terminée. Résultats enregistrés dans chi2_results_bh_corrected_D0.csv


# D3 - données non normalisées avec D60

In [68]:
# Charger le fichier Excel
file_path = "/Users/loictalignani/research/project/lipidomics/data/lipidsig_datasets/somme_des_espèces_lipides_D3.xlsx"
xls = pd.ExcelFile(file_path)

# Charger les données des onglets
sum_species = pd.read_excel(xls, sheet_name="sum_species")
patient_data = pd.read_excel(xls, sheet_name="patient_data")

In [69]:
sum_species

,Family,BS-064D03,BS-082D03,BS-336D03,BS-346D03,BS-351D03,BS-364D03,BS-377D03,BS-671D03,JV-035D03,...,KT-695D03,KT-705D03,KT-716D03,KT-718D03,KT-723D03,KT-771D03,KT-805D03,KT-880D03,KT-926D03,KT-974D03
0,DG,19.968157,3.117386,16.2149,27.207331,15.631862,17.809934,25.295867,30.261837,21.21604,...,12.142673,8.044282,19.705065,14.141924,25.24184,23.197631,18.775687,8.482933,12.441529,10.414962
1,CAR,0.0,0.0,0.677716,0.708692,0.140242,0.568899,0.222659,0.0,1.844806,...,0.135259,0.43282,0.464531,0.784272,0.741406,0.477466,0.221529,0.0,0.0,0.817418
2,CE,2.848492,0.903338,5.007647,11.814341,6.24188,0.0,0.0,3.369775,3.61475,...,3.825023,6.26406,0.0,5.023619,0.0,0.552063,0.0,0.493274,4.419864,8.515708
3,Cer,4.020806,1.348513,1.793138,7.285975,3.109885,6.035633,0.734463,4.252049,1.172186,...,4.505105,1.731756,2.391915,3.32658,1.361323,3.679226,3.345797,1.168313,5.639939,2.048494
4,Chol,8.622505,3.158353,7.555499,11.876356,7.460192,10.01639,0.0,11.772541,0.0,...,7.479568,11.692061,10.578816,8.876283,5.607217,7.47221,4.415338,2.761311,10.559197,4.831007
5,HexCer,1.200005,0.0,0.995126,0.0,2.132215,0.0,0.0,3.287508,0.0,...,0.0,0.0,1.209474,0.0,0.0,0.981445,0.0,0.0,2.483256,2.44228
6,LPC,3055.235935,1382.943091,2916.41176,2366.958261,2663.136456,2047.918692,3347.278615,2905.570574,3680.441973,...,1522.272879,2603.648706,3725.147573,3352.472096,2337.586571,5423.957621,3231.382883,326.267818,1951.959343,2498.226063
7,PC,1291.379157,487.301511,963.20134,2022.505415,1142.050398,1674.839766,1181.767303,1877.201588,1311.075399,...,990.865207,987.840208,1176.458917,938.548077,994.993911,1057.885482,963.521558,774.630985,903.68168,668.41843
8,PE,6.649396,0.0,0.0,15.422787,0.0,7.623165,0.0,10.557481,0.0,...,5.838379,0.0,0.0,0.0,8.899761,13.340312,0.0,0.0,0.0,4.917157
9,SM,246.5139,140.63322,232.841977,267.130409,258.659715,245.841702,179.422754,300.452895,216.264006,...,265.940885,260.031784,309.759924,294.971223,316.908308,414.619134,258.094079,127.98667,348.684308,217.820758


In [70]:
patient_data

,Patient code,Age,Sex,severity
0,BS-064,14,M,severe
1,JV-035,13,M,severe
2,JV-048,12,M,severe
3,KT-515,29,M,severe
4,KT-926,8,M,severe
5,JV-148,14,M,severe
6,JV-157,13,M,severe
7,KT-193,10,M,severe
8,KT-247,10,F,severe
9,KT-312,14,M,severe


In [71]:
# Extraire les informations des patients à partir des colonnes
# Exclure la colonne "Family" et "severity"
sample_cols = sum_species.columns[1:-1]
patient_info = pd.DataFrame({
    # Retirer le jour (D0, D3...) pour matcher patient_data
    "Sample": [col[:-3] for col in sample_cols],
    "Day": [col[-3:] for col in sample_cols],
    "Column": sample_cols,
    # Dernière ligne contient la sévérité
    "Severity": sum_species.iloc[-1, 1:-1].values
})

In [72]:
patient_info

,Sample,Day,Column,Severity
0,BS-064,D03,BS-064D03,severe
1,BS-082,D03,BS-082D03,mild
2,BS-336,D03,BS-336D03,mild
3,BS-346,D03,BS-346D03,mild
4,BS-351,D03,BS-351D03,mild
5,BS-364,D03,BS-364D03,mild
6,BS-377,D03,BS-377D03,mild
7,BS-671,D03,BS-671D03,mild
8,JV-035,D03,JV-035D03,severe
9,JV-138,D03,JV-138D03,mild


In [73]:
# Fusionner avec patient_data pour récupérer le sexe
patient_data = patient_data.rename(columns={"Patient code": "Sample"})
patient_info = patient_info.merge(patient_data, on="Sample", how="left")

In [74]:
patient_info.set_index("Column", inplace=True)
patient_info

,Sample,Day,Severity,Age,Sex,severity
Column,,,,,,
BS-064D03,BS-064,D03,severe,14,M,severe
BS-082D03,BS-082,D03,mild,18,M,mild
BS-336D03,BS-336,D03,mild,11,F,mild
BS-346D03,BS-346,D03,mild,8,F,mild
BS-351D03,BS-351,D03,mild,7,M,mild
BS-364D03,BS-364,D03,mild,6,M,mild
BS-377D03,BS-377,D03,mild,13,F,mild
BS-671D03,BS-671,D03,mild,8,F,mild
JV-035D03,JV-035,D03,severe,13,M,severe


In [75]:
# Appliquer la transformation log2(x + 0.01) sur les valeurs de lipides
lipid_data = sum_species.iloc[:-1, 1:-1].map(lambda x: np.log2(x + 0.01))
lipid_data

,BS-064D03,BS-082D03,BS-336D03,BS-346D03,BS-351D03,BS-364D03,BS-377D03,BS-671D03,JV-035D03,JV-138D03,...,KT-663D03,KT-695D03,KT-705D03,KT-716D03,KT-718D03,KT-723D03,KT-771D03,KT-805D03,KT-880D03,KT-926D03
0,4.320352,1.644958,4.020138,4.766454,3.967340,4.155420,4.661400,4.919904,4.407763,3.810856,...,3.537231,3.603202,3.009756,4.301227,3.822926,4.658317,4.536527,4.231562,3.086263,3.638251
1,-6.643856,-6.643856,-0.540114,-0.476554,-2.734639,-0.788617,-2.103709,-6.643856,0.891268,-6.643856,...,-6.643856,-2.783303,-1.175208,-1.075425,-0.332295,-0.412335,-1.036626,-2.110738,-6.643856,-6.643856
2,1.515254,-0.130779,2.327011,3.563688,2.644290,-6.643856,-6.643856,1.756927,1.857881,-6.643856,...,-1.040649,1.939235,2.649399,-6.643856,2.331596,-6.643856,-0.831195,-6.643856,-0.990585,2.147262
3,2.011068,0.442028,0.850510,2.867101,1.641493,2.595893,-0.425727,2.091547,0.241457,1.518062,...,1.056041,2.174760,0.800543,1.264185,1.738370,0.455569,1.883318,1.746655,0.236723,2.498235
4,3.109779,1.663733,2.919435,3.571235,2.901145,3.325730,-6.643856,3.558579,-6.643856,2.989791,...,3.197970,2.904882,3.548691,3.404469,3.151580,2.489856,2.903464,2.145788,1.470569,3.401794
5,0.275013,-6.643856,0.007377,-6.643856,1.099103,-6.643856,-6.643856,1.721376,-6.643856,0.414607,...,-6.643856,-6.643856,-6.643856,0.286259,-6.643856,-6.643856,-0.012395,-6.643856,-6.643856,1.318031
6,11.577073,10.433537,11.509984,11.208825,11.378916,10.999950,11.708777,11.504611,11.845667,10.574487,...,8.569690,10.572021,11.346325,11.863086,11.711014,11.190810,12.405133,11.657940,8.349957,10.930715
7,10.334708,8.928700,9.911709,10.981935,10.157423,10.709816,10.206742,10.874376,10.356546,9.968235,...,10.141553,9.952560,9.948148,10.200247,9.874302,9.958558,10.046981,9.912188,9.597384,9.819687
8,2.735391,-6.643850,-6.643854,3.947927,-6.643853,2.932281,-6.643854,3.401560,-6.643852,2.041658,...,4.302206,2.548037,-6.643853,-6.643854,-6.643854,3.155387,3.738802,-6.643853,-6.643854,-6.643854
9,7.945584,7.135896,7.863269,8.061454,8.014967,7.941645,7.487299,8.231043,7.756716,7.827887,...,8.756047,8.055016,8.022600,8.275053,8.204479,8.307967,8.695678,8.011809,6.999962,8.445819


In [76]:
# Déterminer un seuil pour chaque famille de lipides (ex : médiane)
thresholds = lipid_data.median(axis=1)
thresholds

0      4.155420
1     -1.988607
2     -0.130779
3      1.441680
4      2.903464
5     -6.643856
6     11.318400
7      9.968235
8     -6.643852
9      7.941645
10     7.066998
11     8.180973
12     4.766097
dtype: float64

In [77]:
# Binariser les valeurs des lipides (1 si supérieur au seuil, 0 sinon)
binary_lipid_data = lipid_data.gt(thresholds, axis=0).astype(int)
binary_lipid_data

,BS-064D03,BS-082D03,BS-336D03,BS-346D03,BS-351D03,BS-364D03,BS-377D03,BS-671D03,JV-035D03,JV-138D03,...,KT-663D03,KT-695D03,KT-705D03,KT-716D03,KT-718D03,KT-723D03,KT-771D03,KT-805D03,KT-880D03,KT-926D03
0,1,0,0,1,0,0,1,1,1,0,...,0,0,0,1,0,1,1,1,0,0
1,0,0,1,1,0,1,0,0,1,0,...,0,0,1,1,1,1,1,0,0,0
2,1,0,1,1,1,0,0,1,1,0,...,0,1,1,0,1,0,0,0,0,1
3,1,0,0,1,1,1,0,1,0,1,...,0,1,0,0,1,0,1,1,0,1
4,1,0,1,1,0,1,0,1,0,1,...,1,1,1,1,1,0,0,0,0,1
5,1,1,1,1,1,1,0,1,1,1,...,0,0,0,1,0,0,1,1,0,1
6,1,0,1,0,1,0,1,1,1,0,...,0,0,1,1,1,0,1,1,0,0
7,1,0,0,1,1,1,1,1,1,0,...,1,0,0,1,0,0,1,0,0,0
8,1,1,0,1,0,1,0,1,0,1,...,1,1,0,0,0,1,1,0,0,0
9,1,0,0,1,1,0,0,1,0,0,...,1,1,1,1,1,1,1,1,0,1


## Test du Chi² sur les lipides en fonction de la sévérité

In [78]:
# Effectuer les tests du Chi²
chi2_results = {}

# Chi² entre la sévérité et la présence des familles de lipides
for i, lipid in enumerate(sum_species.iloc[:-1, 0]):
    contingency_table = pd.crosstab(
        binary_lipid_data.iloc[i, :], patient_info["Severity"])
    chi2, p, _, _ = chi2_contingency(contingency_table)
    chi2_results[lipid] = {"Chi2": chi2, "p-value": p}

In [79]:
chi2_results

{'DG': {'Chi2': 0.22053571428571378, 'p-value': 0.6386320338113531},
 'CAR': {'Chi2': 0.22053571428571378, 'p-value': 0.6386320338113531},
 'CE': {'Chi2': 1.2370770676691718, 'p-value': 0.2660351186466848},
 'Cer': {'Chi2': 0.0, 'p-value': 1.0},
 'Chol': {'Chi2': 0.029934210526315973, 'p-value': 0.862639591019358},
 'HexCer': {'Chi2': 2.126550751879701, 'p-value': 0.14476619644361308},
 'LPC': {'Chi2': 1.2370770676691718, 'p-value': 0.2660351186466848},
 'PC': {'Chi2': 0.0, 'p-value': 1.0},
 'PE': {'Chi2': 0.029934210526315973, 'p-value': 0.862639591019358},
 'SM': {'Chi2': 0.029934210526315973, 'p-value': 0.862639591019358},
 'TG': {'Chi2': 3.0795582706766904, 'p-value': 0.07928205591587303},
 'FA': {'Chi2': 0.22053571428571378, 'p-value': 0.6386320338113531},
 'PI': {'Chi2': 1.2370770676691718, 'p-value': 0.2660351186466848}}

Aucun lipide ne montre une association statistiquement significative avec la sévérité de la dengue. Cela suggère que la présence (ou l’absence) de ces lipides ne seraient pas liés à la gravité de la maladie.

In [86]:
# Convertir les résultats en DataFrame et sauvegarder
chi2_df = pd.DataFrame.from_dict(chi2_results, orient="index")


In [87]:
chi2_df.to_csv(
    "/Users/loictalignani/research/project/lipidomics/analysis/univariate_analysis/chi2_results_D3.csv", index=True)

print("Analyse Chi² terminée. Résultats enregistrés dans chi2_results_D3.csv")

Analyse Chi² terminée. Résultats enregistrés dans chi2_results_D3.csv


## Benjamini-Hochberg correction
Plusieurs tests en parallèle sont effectués, ce qui augmente le risque de faux positifs. La correction BH ajuste les p-values pour tenir compte de ce biais. Elle contrôle le taux de fausses découvertes (FDR) plutôt que le taux d’erreur global. C’est plus adapté que Bonferroni, qui est trop strict. Les p-values corrigées sont souvent plus grandes, ce qui diminue le risque d’interpréter à tort un résultat comme significatif.

In [88]:
from statsmodels.stats.multitest import multipletests

In [89]:
# Correction de Benjamini-Hochberg pour toutes les p-values
_, corrected_p_values, _, _ = multipletests(
    chi2_df["p-value"], method="fdr_bh")

# Ajouter les p-values corrigées au DataFrame
chi2_df["p-value (BH corrected)"] = corrected_p_values

# Afficher les résultats
print(chi2_df)

                     Chi2   p-value  p-value (BH corrected)
DG               0.220536  0.638632                0.993428
CAR              0.220536  0.638632                0.993428
CE               1.237077  0.266035                0.620749
Cer              0.000000  1.000000                1.000000
Chol             0.029934  0.862640                1.000000
HexCer           2.126551  0.144766                0.620749
LPC              1.237077  0.266035                0.620749
PC               0.000000  1.000000                1.000000
PE               0.029934  0.862640                1.000000
SM               0.029934  0.862640                1.000000
TG               3.079558  0.079282                0.619370
FA               0.220536  0.638632                0.993428
PI               1.237077  0.266035                0.620749
Sex vs Severity  2.901786  0.088481                0.619370


In [90]:
chi2_df.to_csv(
    "/Users/loictalignani/research/project/lipidomics/analysis/univariate_analysis/chi2_results_bh_corrected_D3.csv", index=True)

print("Analyse Chi² terminée. Résultats enregistrés dans chi2_results_bh_corrected_D3.csv")

Analyse Chi² terminée. Résultats enregistrés dans chi2_results_bh_corrected_D3.csv


# D10 - données non normalisées avec D60

In [91]:
# Charger le fichier Excel
file_path = "/Users/loictalignani/research/project/lipidomics/data/lipidsig_datasets/somme_des_espèces_lipides_D10.xlsx"
xls = pd.ExcelFile(file_path)

# Charger les données des onglets
sum_species = pd.read_excel(xls, sheet_name="sum_species")
patient_data = pd.read_excel(xls, sheet_name="patient_data")

In [92]:
sum_species

,Family,BS-064D10,BS-082D10,BS-336D10,BS-346D10,BS-351D10,BS-364D10,BS-377D10,BS-671D10,JV-035D10,...,KT-695D10,KT-705D10,KT-716D10,KT-718D10,KT-723D10,KT-771D10,KT-805D10,KT-880D10,KT-926D10,KT-974D10
0,DG,17.14387,8.932488,16.818381,43.06756,20.678015,17.238723,22.828113,10.631183,31.610864,...,15.974618,12.078603,4.522523,24.089838,21.538093,29.826527,39.65641,27.917212,12.63445,7.077127
1,CAR,0.0,0.157989,0.13242,0.587202,0.0,0.876243,0.12918,0.139246,0.0,...,0.562671,0.241541,0.09994,0.112721,0.0,0.242009,0.125143,0.333768,0.0,0.0
2,CE,0.0,3.49802,5.680665,0.0,0.818111,3.075995,0.596266,1.50148,0.0,...,0.492665,1.10157,4.74578,3.433631,4.235984,4.140759,2.38146,0.0,0.0,0.0
3,Cer,2.69399,1.487014,2.639436,2.032787,2.761311,1.98128,1.985246,1.4586,1.492553,...,2.293471,2.2072,1.831182,3.825734,4.000152,2.114922,2.276248,1.794931,2.856815,2.011308
4,Chol,7.239538,3.166453,6.545276,8.051377,6.853948,7.724413,5.896456,9.15952,0.0,...,5.212779,8.712769,10.467162,12.170495,9.741615,11.937205,4.308602,8.229098,4.371859,5.124208
5,HexCer,0.0,0.0,1.181572,0.0,0.0,1.351111,0.0,0.0,0.0,...,0.0,0.0,0.0,0.747076,0.936774,0.724351,0.0,0.0,0.0,0.0
6,LPC,5249.176174,2382.796609,4491.96769,5931.872054,3355.32854,5447.62691,3905.996284,4447.933695,3203.237789,...,5987.756741,4549.63197,3914.471619,5134.018605,4351.614278,4693.378181,4365.95225,2537.055866,3808.740103,3253.496498
7,PC,1853.936574,696.448882,1496.670193,2685.324364,1423.513974,1445.205676,1564.542329,1014.618086,1554.976813,...,1354.029404,1764.619306,1345.198729,2243.03347,1047.403808,1673.373085,1183.84809,2104.073344,1042.136267,1115.957729
8,PE,9.021984,0.0,0.0,13.728119,0.0,0.0,9.885526,0.0,0.0,...,5.371785,5.496055,9.002522,13.799677,0.0,29.284946,0.0,14.831093,0.0,0.0
9,SM,242.426089,107.55478,228.352511,240.467079,174.398995,259.471558,182.578614,257.435066,191.389987,...,189.611506,250.551399,279.93824,278.262692,231.335389,296.21825,181.289846,352.295045,192.032251,181.889737


In [93]:
patient_data

,Patient code,Age,Sex,severity
0,BS-064,14,M,severe
1,JV-035,13,M,severe
2,JV-048,12,M,severe
3,KT-515,29,M,severe
4,KT-926,8,M,severe
5,JV-148,14,M,severe
6,JV-157,13,M,severe
7,KT-193,10,M,severe
8,KT-247,10,F,severe
9,KT-312,14,M,severe


In [94]:
# Extraire les informations des patients à partir des colonnes
# Exclure la colonne "Family" et "severity"
sample_cols = sum_species.columns[1:-1]
patient_info = pd.DataFrame({
    # Retirer le jour (D0, D3...) pour matcher patient_data
    "Sample": [col[:-3] for col in sample_cols],
    "Day": [col[-3:] for col in sample_cols],
    "Column": sample_cols,
    # Dernière ligne contient la sévérité
    "Severity": sum_species.iloc[-1, 1:-1].values
})

In [95]:
patient_info

,Sample,Day,Column,Severity
0,BS-064,D10,BS-064D10,severe
1,BS-082,D10,BS-082D10,mild
2,BS-336,D10,BS-336D10,mild
3,BS-346,D10,BS-346D10,mild
4,BS-351,D10,BS-351D10,mild
5,BS-364,D10,BS-364D10,mild
6,BS-377,D10,BS-377D10,mild
7,BS-671,D10,BS-671D10,mild
8,JV-035,D10,JV-035D10,severe
9,JV-138,D10,JV-138D10,mild


In [96]:
# Fusionner avec patient_data pour récupérer le sexe
patient_data = patient_data.rename(columns={"Patient code": "Sample"})
patient_info = patient_info.merge(patient_data, on="Sample", how="left")

In [97]:
patient_info.set_index("Column", inplace=True)
patient_info

,Sample,Day,Severity,Age,Sex,severity
Column,,,,,,
BS-064D10,BS-064,D10,severe,14,M,severe
BS-082D10,BS-082,D10,mild,18,M,mild
BS-336D10,BS-336,D10,mild,11,F,mild
BS-346D10,BS-346,D10,mild,8,F,mild
BS-351D10,BS-351,D10,mild,7,M,mild
BS-364D10,BS-364,D10,mild,6,M,mild
BS-377D10,BS-377,D10,mild,13,F,mild
BS-671D10,BS-671,D10,mild,8,F,mild
JV-035D10,JV-035,D10,severe,13,M,severe


In [98]:
# Appliquer la transformation log2(x + 0.01) sur les valeurs de lipides
lipid_data = sum_species.iloc[:-1, 1:-1].map(lambda x: np.log2(x + 0.01))
lipid_data

,BS-064D10,BS-082D10,BS-336D10,BS-346D10,BS-351D10,BS-364D10,BS-377D10,BS-671D10,JV-035D10,JV-138D10,...,KT-663D10,KT-695D10,KT-705D10,KT-716D10,KT-718D10,KT-723D10,KT-771D10,KT-805D10,KT-880D10,KT-926D10
0,4.100462,3.160676,4.072824,5.428865,4.370723,4.108418,4.513372,3.411587,4.982805,5.349059,...,4.695467,3.998612,3.595576,2.180314,4.590952,4.429488,4.899008,5.309846,4.803600,3.660432
1,-6.643856,-2.573561,-2.811781,-0.743708,-6.643856,-0.174227,-2.844978,-2.744239,-6.643856,-6.643856,...,-6.643856,-0.804222,-1.991133,-3.185213,-3.026550,-6.643856,-1.988454,-2.887445,-1.540491,-6.643856
2,-6.643856,1.810657,2.508597,-6.643856,-0.272105,1.625736,-0.721977,0.595962,-6.643856,-0.211547,...,-6.643856,-0.992330,0.152598,2.249682,1.783931,2.086099,2.053375,1.257892,-6.643856,-6.643856
3,1.435090,0.582087,1.405685,1.030539,1.470569,0.993696,0.996566,0.554441,0.587416,1.815228,...,2.229638,1.203810,1.148739,0.880632,1.939503,2.003657,1.087410,1.192982,0.851944,1.519449
4,2.857889,1.667417,2.712656,3.011026,2.779039,2.951292,2.562293,3.196846,-6.643856,2.604255,...,3.807534,2.384818,3.124786,3.389176,3.606501,3.285641,3.578601,2.110564,3.042486,2.131543
5,-6.643856,-6.643856,0.252866,-6.643856,-6.643856,0.444785,-6.643856,-6.643856,-6.643856,1.060255,...,-6.643856,-6.643856,-6.643856,-6.643856,-0.401490,-0.078908,-0.445458,-6.643856,-6.643856,-6.643856
6,12.357878,11.218446,12.133135,12.534274,11.712243,12.411415,11.931479,12.118923,11.645320,11.992921,...,12.453829,12.547802,12.151537,11.934606,12.325876,12.087338,12.196414,12.092084,11.308945,11.895102
7,10.856384,9.443894,10.547550,11.390886,10.475251,10.497069,10.611534,9.986735,10.602687,11.117754,...,10.632331,10.403054,10.785149,10.393614,11.131242,10.032616,10.708552,10.209280,11.038976,10.025342
8,3.175043,-6.643854,-6.643854,3.780113,-6.643854,-6.643853,3.306776,-6.643853,-6.643852,4.491304,...,-6.643853,2.428085,2.461019,3.171931,3.787608,-6.643854,4.872580,-6.643854,3.891525,-6.643853
9,7.921461,6.749062,7.835182,7.909756,7.446331,8.019488,7.512453,8.008121,7.580447,7.939963,...,8.028507,7.566979,7.969020,8.129016,8.120356,7.853905,8.210565,7.502234,8.460681,7.585280


In [99]:
# Déterminer un seuil pour chaque famille de lipides (ex : médiane)
thresholds = lipid_data.median(axis=1)
thresholds

0      4.331233
1     -3.185213
2      0.548317
3      1.127642
4      2.891119
5     -6.643856
6     12.133135
7     10.547550
8     -6.643851
9      7.835182
10     7.022551
11     7.431276
12     5.634490
dtype: float64

In [100]:
# Binariser les valeurs des lipides (1 si supérieur au seuil, 0 sinon)
binary_lipid_data = lipid_data.gt(thresholds, axis=0).astype(int)
binary_lipid_data

,BS-064D10,BS-082D10,BS-336D10,BS-346D10,BS-351D10,BS-364D10,BS-377D10,BS-671D10,JV-035D10,JV-138D10,...,KT-663D10,KT-695D10,KT-705D10,KT-716D10,KT-718D10,KT-723D10,KT-771D10,KT-805D10,KT-880D10,KT-926D10
0,0,0,0,1,1,0,1,0,1,1,...,1,0,0,0,1,1,1,1,1,0
1,0,1,1,1,0,1,1,1,0,0,...,0,1,1,0,1,0,1,1,1,0
2,0,1,1,0,0,1,0,1,0,0,...,0,0,0,1,1,1,1,1,0,0
3,1,0,1,0,1,0,0,0,0,1,...,1,1,1,0,1,1,0,1,0,1
4,0,0,0,1,0,1,0,1,0,0,...,1,0,1,1,1,1,1,0,1,0
5,1,0,1,1,0,1,0,1,1,1,...,1,0,0,0,1,1,1,0,1,1
6,1,0,0,1,0,1,0,0,0,0,...,1,1,1,0,1,0,1,0,0,0
7,1,0,0,1,0,0,1,0,1,1,...,1,0,1,0,1,0,1,0,1,0
8,1,0,0,1,0,0,1,0,0,1,...,0,1,1,1,1,0,1,0,1,0
9,1,0,0,1,0,1,0,1,0,1,...,1,0,1,1,1,1,1,0,1,0


## Test du Chi² sur les lipides en fonction de la sévérité

In [101]:
# Effectuer les tests du Chi²
chi2_results = {}

# Chi² entre la sévérité et la présence des familles de lipides
for i, lipid in enumerate(sum_species.iloc[:-1, 0]):
    contingency_table = pd.crosstab(
        binary_lipid_data.iloc[i, :], patient_info["Severity"])
    chi2, p, _, _ = chi2_contingency(contingency_table)
    chi2_results[lipid] = {"Chi2": chi2, "p-value": p}

In [102]:
chi2_results

{'DG': {'Chi2': 0.029934210526315973, 'p-value': 0.862639591019358},
 'CAR': {'Chi2': 2.126550751879701, 'p-value': 0.14476619644361308},
 'CE': {'Chi2': 0.0, 'p-value': 1.0},
 'Cer': {'Chi2': 1.2370770676691718, 'p-value': 0.2660351186466848},
 'Chol': {'Chi2': 0.6652725563909783, 'p-value': 0.41470468054923515},
 'HexCer': {'Chi2': 0.0, 'p-value': 1.0},
 'LPC': {'Chi2': 0.22053571428571378, 'p-value': 0.6386320338113531},
 'PC': {'Chi2': 1.2370770676691718, 'p-value': 0.2660351186466848},
 'PE': {'Chi2': 0.029934210526315973, 'p-value': 0.862639591019358},
 'SM': {'Chi2': 0.029934210526315973, 'p-value': 0.862639591019358},
 'TG': {'Chi2': 0.029934210526315973, 'p-value': 0.862639591019358},
 'FA': {'Chi2': 0.6652725563909783, 'p-value': 0.41470468054923515},
 'PI': {'Chi2': 0.22053571428571378, 'p-value': 0.6386320338113531}}

Aucun lipide ne montre une association statistiquement significative avec la sévérité de la dengue. Cela suggère que la présence (ou l’absence) de ces lipides ne seraient pas liés à la gravité de la maladie.

In [103]:
# Convertir les résultats en DataFrame et sauvegarder
chi2_df = pd.DataFrame.from_dict(chi2_results, orient="index")


In [104]:
chi2_df.to_csv(
    "/Users/loictalignani/research/project/lipidomics/analysis/univariate_analysis/chi2_results_D10.csv", index=True)

print("Analyse Chi² terminée. Résultats enregistrés dans chi2_results_D10.csv")

Analyse Chi² terminée. Résultats enregistrés dans chi2_results_D10.csv


## Benjamini-Hochberg correction
Plusieurs tests en parallèle sont effectués, ce qui augmente le risque de faux positifs. La correction BH ajuste les p-values pour tenir compte de ce biais. Elle contrôle le taux de fausses découvertes (FDR) plutôt que le taux d’erreur global. C’est plus adapté que Bonferroni, qui est trop strict. Les p-values corrigées sont souvent plus grandes, ce qui diminue le risque d’interpréter à tort un résultat comme significatif.

In [105]:
from statsmodels.stats.multitest import multipletests

In [106]:
# Correction de Benjamini-Hochberg pour toutes les p-values
_, corrected_p_values, _, _ = multipletests(
    chi2_df["p-value"], method="fdr_bh")

# Ajouter les p-values corrigées au DataFrame
chi2_df["p-value (BH corrected)"] = corrected_p_values

# Afficher les résultats
print(chi2_df)

            Chi2   p-value  p-value (BH corrected)
DG      0.029934  0.862640                     1.0
CAR     2.126551  0.144766                     1.0
CE      0.000000  1.000000                     1.0
Cer     1.237077  0.266035                     1.0
Chol    0.665273  0.414705                     1.0
HexCer  0.000000  1.000000                     1.0
LPC     0.220536  0.638632                     1.0
PC      1.237077  0.266035                     1.0
PE      0.029934  0.862640                     1.0
SM      0.029934  0.862640                     1.0
TG      0.029934  0.862640                     1.0
FA      0.665273  0.414705                     1.0
PI      0.220536  0.638632                     1.0


In [107]:
chi2_df.to_csv(
    "/Users/loictalignani/research/project/lipidomics/analysis/univariate_analysis/chi2_results_bh_corrected_D10.csv", index=True)

print("Analyse Chi² terminée. Résultats enregistrés dans chi2_results_bh_corrected_D10.csv")

Analyse Chi² terminée. Résultats enregistrés dans chi2_results_bh_corrected_D10.csv


# D0 - données normalisées (ratio D0/D60)

In [ ]:
# Charger le fichier Excel
file_path = "/Users/loictalignani/research/project/lipidomics/data/lipidsig_datasets/Ratios/Ratio_lipid_abundance_data_D0_D60.xlsx"
xls = pd.ExcelFile(file_path)

# Charger les données des onglets
sum_species = pd.read_excel(xls, sheet_name="species")
patient_data = pd.read_excel(xls, sheet_name="patient_data")

In [3]:
sum_species

,feature,KT-539_D0_D60,BS-671_D0_D60,BS-064_D0_D60,KT-805_D0_D60,KT-926_D0_D60,KT-705_D0_D60,BS-336_D0_D60,BS-082_D0_D60,BS-351_D0_D60,...,KT-974_D0_D60,KT-537_D0_D60,KT-612_D0_D60,KT-445_D0_D60,KT-525_D0_D60,KT-352_D0_D60,JV-148_D0_D60,BS-346_D0_D60,KT-663_D0_D60,KT-695_D0_D60
0,DG 30:0,1.000000,1.000000,1.000000,1.000000,1.000000,0.699301,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
1,DG 32:0,1.000000,0.602410,0.897436,1.063953,1.546763,0.770492,0.684932,1.013245,1.006803,...,0.675676,1.000000,0.717514,0.621118,1.000000,1.430000,0.729927,0.458716,0.694444,1.000000
2,DG 32:1,1.000000,1.000000,1.000000,1.039735,2.150000,0.431034,0.632911,0.925676,1.000000,...,1.000000,1.000000,0.617284,0.662252,1.000000,1.000000,0.740741,0.606061,1.000000,1.000000
3,DG 34:1,0.708812,0.614078,0.564767,1.330661,3.283636,0.863551,0.554865,1.223022,0.914040,...,0.526611,1.185185,0.386667,0.503597,1.788945,1.202346,1.384342,0.259414,0.506410,0.757576
4,DG 34:3,1.000000,1.000000,1.000000,1.000000,1.690000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.862069,1.000000,1.000000,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159,PI 34:2,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,0.318471,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
160,PI 36:2,1.000000,0.165017,0.187970,1.090909,3.200000,0.658307,0.158479,1.000000,0.722222,...,1.000000,1.000000,0.245700,0.133869,5.130000,7.930000,1.634752,0.152439,0.241546,1.000000
161,PI 36:4,1.000000,1.000000,1.000000,0.255754,4.630000,0.215517,1.000000,1.000000,0.215983,...,1.000000,1.000000,1.000000,0.245098,1.000000,1.000000,0.416667,1.000000,1.000000,1.000000
162,PI 38:3,1.000000,1.000000,1.000000,0.214159,1.520000,0.444444,0.458716,0.793651,1.000000,...,0.613497,1.000000,1.000000,0.632911,1.000000,1.330000,0.595238,1.000000,1.000000,1.000000


In [4]:
patient_data

,Patient code,Age,Sex,severity
0,BS-064,14,M,severe
1,JV-035,13,M,severe
2,KT-515,29,M,severe
3,KT-926,8,M,severe
4,JV-148,14,M,severe
5,JV-157,13,M,severe
6,KT-193,10,M,severe
7,KT-247,10,F,severe
8,KT-312,14,M,severe
9,KT-347,9,M,severe


In [6]:
sample_cols = sum_species.columns[1:]
sample_cols

Index(['KT-539_D0_D60', 'BS-671_D0_D60', 'BS-064_D0_D60', 'KT-805_D0_D60',
       'KT-926_D0_D60', 'KT-705_D0_D60', 'BS-336_D0_D60', 'BS-082_D0_D60',
       'BS-351_D0_D60', 'KT-347_D0_D60', 'BS-377_D0_D60', 'KT-313_D0_D60',
       'BS-364_D0_D60', 'KT-880_D0_D60', 'JV-138_D0_D60', 'KT-412_D0_D60',
       'KT-515_D0_D60', 'KT-771_D0_D60', 'KT-723_D0_D60', 'KT-718_D0_D60',
       'KT-417_D0_D60', 'KT-522_D0_D60', 'KT-312_D0_D60', 'JV-157_D0_D60',
       'KT-538_D0_D60', 'KT-193_D0_D60', 'JV-035_D0_D60', 'KT-434_D0_D60',
       'KT-247_D0_D60', 'KT-716_D0_D60', 'KT-974_D0_D60', 'KT-537_D0_D60',
       'KT-612_D0_D60', 'KT-445_D0_D60', 'KT-525_D0_D60', 'KT-352_D0_D60',
       'JV-148_D0_D60', 'BS-346_D0_D60', 'KT-663_D0_D60', 'KT-695_D0_D60'],
      dtype='object')

In [7]:
# Extraire les informations des patients à partir des colonnes
# Exclure la colonne "Family" et "severity"
sample_cols = sum_species.columns[1:]
patient_info = pd.DataFrame({
    # Retirer le jour (D0, D3...) pour matcher patient_data
    "Sample": [col[:6] for col in sample_cols],
    "Day": [col[7:9] for col in sample_cols],
    "Column": sample_cols,
    # Dernière ligne contient la sévérité
    "Severity": sum_species.iloc[-1, 1:].values
})
patient_info

,Sample,Day,Column,Severity
0,KT-539,D0,KT-539_D0_D60,0.032062
1,BS-671,D0,BS-671_D0_D60,0.566755
2,BS-064,D0,BS-064_D0_D60,0.957614
3,KT-805,D0,KT-805_D0_D60,1.860165
4,KT-926,D0,KT-926_D0_D60,0.52917
5,KT-705,D0,KT-705_D0_D60,0.402533
6,BS-336,D0,BS-336_D0_D60,0.392225
7,BS-082,D0,BS-082_D0_D60,0.294778
8,BS-351,D0,BS-351_D0_D60,0.476975
9,KT-347,D0,KT-347_D0_D60,0.311385


In [8]:
# Fusionner avec patient_data pour récupérer le sexe
patient_data = patient_data.rename(columns={"Patient code": "Sample"})
patient_info = patient_info.merge(patient_data, on="Sample", how="left")
patient_info.set_index("Column", inplace=True)
patient_info

,Sample,Day,Severity,Age,Sex,severity
Column,,,,,,
KT-539_D0_D60,KT-539,D0,0.032062,5,M,mild
BS-671_D0_D60,BS-671,D0,0.566755,8,F,mild
BS-064_D0_D60,BS-064,D0,0.957614,14,M,severe
KT-805_D0_D60,KT-805,D0,1.860165,13,M,mild
KT-926_D0_D60,KT-926,D0,0.52917,8,M,severe
KT-705_D0_D60,KT-705,D0,0.402533,17,M,severe
BS-336_D0_D60,BS-336,D0,0.392225,11,F,mild
BS-082_D0_D60,BS-082,D0,0.294778,18,M,mild
BS-351_D0_D60,BS-351,D0,0.476975,7,M,mild


In [14]:
lipid_data = sum_species.iloc[:, 1:]
print(lipid_data)
lipid_data = lipid_data.astype(float)

     KT-539_D0_D60  BS-671_D0_D60  BS-064_D0_D60  KT-805_D0_D60  \
0         1.000000       1.000000       1.000000       1.000000   
1         1.000000       0.602410       0.897436       1.063953   
2         1.000000       1.000000       1.000000       1.039735   
3         0.708812       0.614078       0.564767       1.330661   
4         1.000000       1.000000       1.000000       1.000000   
..             ...            ...            ...            ...   
159       1.000000       1.000000       1.000000       1.000000   
160       1.000000       0.165017       0.187970       1.090909   
161       1.000000       1.000000       1.000000       0.255754   
162       1.000000       1.000000       1.000000       0.214159   
163       0.032062       0.566755       0.957614       1.860165   

     KT-926_D0_D60  KT-705_D0_D60  BS-336_D0_D60  BS-082_D0_D60  \
0         1.000000       0.699301       1.000000       1.000000   
1         1.546763       0.770492       0.684932       1.0132

In [15]:
# Déterminer un seuil pour chaque lipides (ex : médiane).
thresholds = lipid_data.median(axis=1)
thresholds

0      1.000000
1      1.000000
2      1.000000
3      0.756983
4      1.000000
         ...   
159    1.000000
160    0.996278
161    1.000000
162    0.916667
163    0.469499
Length: 164, dtype: float64

In [16]:
# Binariser les valeurs des lipides (1 si supérieur au seuil, 0 sinon)
binary_lipid_data = lipid_data.gt(thresholds, axis=0).astype(int)
binary_lipid_data

,KT-539_D0_D60,BS-671_D0_D60,BS-064_D0_D60,KT-805_D0_D60,KT-926_D0_D60,KT-705_D0_D60,BS-336_D0_D60,BS-082_D0_D60,BS-351_D0_D60,KT-347_D0_D60,...,KT-974_D0_D60,KT-537_D0_D60,KT-612_D0_D60,KT-445_D0_D60,KT-525_D0_D60,KT-352_D0_D60,JV-148_D0_D60,BS-346_D0_D60,KT-663_D0_D60,KT-695_D0_D60
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,1,1,0,0,1,1,1,...,0,0,0,0,0,1,0,0,0,0
2,0,0,0,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,1,1,1,0,1,1,1,...,0,1,0,0,1,1,1,0,0,1
4,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
160,1,0,0,1,1,0,0,1,0,0,...,1,1,0,0,1,1,1,0,0,1
161,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
162,1,1,1,0,1,0,0,0,1,0,...,0,1,1,0,1,1,0,1,1,1


## Test du Chi² sur les lipides en fonction de la sévérité

In [17]:
# Effectuer les tests du Chi²
chi2_results = {}

# Chi² entre la sévérité et la présence des familles de lipides
for i, lipid in enumerate(sum_species.iloc[:-1, 0]):
    contingency_table = pd.crosstab(
        binary_lipid_data.iloc[i, :], patient_info["severity"])
    chi2, p, _, _ = chi2_contingency(contingency_table)
    chi2_results[lipid] = {"Chi2": chi2, "p-value": p}
chi2_results

{'DG 30:0': {'Chi2': 0.010360010360010372, 'p-value': 0.9189280178118044},
 'DG 32:0': {'Chi2': 4.546564546564547, 'p-value': 0.03298486499645653},
 'DG 32:1': {'Chi2': 1.443001443001443, 'p-value': 0.22965425863747851},
 'DG 34:1': {'Chi2': 4.94949494949495, 'p-value': 0.026098291541307057},
 'DG 34:3': {'Chi2': 0.76555023923445, 'p-value': 0.38159715132404115},
 'DG 36:3': {'Chi2': 2.525252525252525, 'p-value': 0.11203684368556334},
 'DG 34:2': {'Chi2': 4.94949494949495, 'p-value': 0.026098291541307057},
 'DG 36:1': {'Chi2': 4.583333333333334, 'p-value': 0.032284353910766866},
 'DG 36:2': {'Chi2': 12.222222222222221, 'p-value': 0.00047223649963052994},
 'DG 36:4': {'Chi2': 0.03276003276003268, 'p-value': 0.8563696732877363},
 'CAR 16:0': {'Chi2': 0.10101010101010101, 'p-value': 0.7506208241656307},
 'CAR 18:1': {'Chi2': 0.10101010101010101, 'p-value': 0.7506208241656307},
 'CAR 18:2': {'Chi2': 2.525252525252525, 'p-value': 0.11203684368556334},
 'CAR 2:0': {'Chi2': 0.0, 'p-value': 1.

DG, TG et PI montrent une association statistiquement significative avec la sévérité de la dengue. Cela suggère que la présence (ou l’absence) de ces lipides seraient liés à la gravité de la maladie. Reste à voir si cette valeur passe la correction de Benjamini-Hochberg.

In [18]:
# Convertir les résultats en DataFrame et sauvegarder
chi2_df = pd.DataFrame.from_dict(chi2_results, orient="index")
chi2_df


,Chi2,p-value
DG 30:0,0.010360,0.918928
DG 32:0,4.546565,0.032985
DG 32:1,1.443001,0.229654
DG 34:1,4.949495,0.026098
DG 34:3,0.765550,0.381597
...,...,...
PI 34:1,0.000000,1.000000
PI 34:2,0.000000,1.000000
PI 36:2,0.101010,0.750621
PI 36:4,0.010360,0.918928


In [19]:
chi2_df.to_csv(
    "/Users/loictalignani/research/project/lipidomics/analysis/univariate_analysis/chi2_results_ratio_D0_D60.csv", index=True)

print("Analyse Chi² terminée. Résultats enregistrés dans chi2_results_ratio_D0_D60.csv")

Analyse Chi² terminée. Résultats enregistrés dans chi2_results_ratio_D0_D60.csv


## Benjamini-Hochberg correction
Plusieurs tests en parallèle sont effectués, ce qui augmente le risque de faux positifs. La correction BH ajuste les p-values pour tenir compte de ce biais. Elle contrôle le taux de fausses découvertes (FDR) plutôt que le taux d’erreur global. C’est plus adapté que Bonferroni, qui est trop strict. Les p-values corrigées sont souvent plus grandes, ce qui diminue le risque d’interpréter à tort un résultat comme significatif.

In [22]:
# Correction de Benjamini-Hochberg pour toutes les p-values
_, corrected_p_values, _, _ = multipletests(
    chi2_df["p-value"], method="fdr_bh")

# Ajouter les p-values corrigées au DataFrame
chi2_df["p-value (BH corrected)"] = corrected_p_values

# Afficher les résultats
print(chi2_df)

             Chi2   p-value  p-value (BH corrected)
DG 30:0  0.010360  0.918928                1.000000
DG 32:0  4.546565  0.032985                0.244388
DG 32:1  1.443001  0.229654                0.759972
DG 34:1  4.949495  0.026098                0.244388
DG 34:3  0.765550  0.381597                0.808891
...           ...       ...                     ...
PI 34:1  0.000000  1.000000                1.000000
PI 34:2  0.000000  1.000000                1.000000
PI 36:2  0.101010  0.750621                1.000000
PI 36:4  0.010360  0.918928                1.000000
PI 38:3  0.909091  0.340356                0.759972

[163 rows x 3 columns]


DG, TG passent avec succès la correction de Benjamini-Hochberg.

In [23]:
chi2_df.to_csv(
    "/Users/loictalignani/research/project/lipidomics/analysis/univariate_analysis/chi2_results_ratio_D0_D60.csv", index=True)

print("Analyse Chi² terminée. Résultats enregistrés dans chi2_results_ratio_D0_D60.csv")

Analyse Chi² terminée. Résultats enregistrés dans chi2_results_ratio_D0_D60.csv


# D3 - données normalisées (ratio D3/D60)

In [43]:
# Charger le fichier Excel
file_path = "/Users/loictalignani/research/project/lipidomics/data/lipidsig_datasets/Ratios/Ratio_lipid_abundance_data_D3_D60.xlsx"
xls = pd.ExcelFile(file_path)

# Charger les données des onglets
sum_species = pd.read_excel(xls, sheet_name="species")
patient_data = pd.read_excel(xls, sheet_name="patient_data")

In [44]:
sum_species

,feature,KT-539_D3_D60,BS-671_D3_D60,BS-064_D3_D60,KT-805_D3_D60,KT-926_D3_D60,KT-705_D3_D60,BS-336_D3_D60,BS-082_D3_D60,BS-351_D3_D60,...,KT-974_D3_D60,KT-537_D3_D60,KT-612_D3_D60,KT-445_D3_D60,KT-525_D3_D60,KT-352_D3_D60,JV-148_D3_D60,BS-346_D3_D60,KT-663_D3_D60,KT-695_D3_D60
0,DG 30:0,1.0,1.0,1.0,1.0,1.0,0.6993006993006994,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1,DG 32:0,1.0,1.86144578313253,0.9128205128205129,1.2209302325581397,1.3741007194244603,0.6188524590163934,1.0136986301369864,0.6622516556291391,1.2789115646258502,...,0.9527027027027026,1.0,1.5875706214689265,0.9813664596273293,1.37,1.37,1.437956204379562,1.1926605504587158,1.1319444444444444,1.95
2,DG 32:1,1.0,1.88,1.8599999999999999,0.6622516556291391,1.37,0.43103448275862066,0.930379746835443,0.6756756756756757,1.5699999999999998,...,1.0,1.0,1.191358024691358,0.8874172185430464,1.0,1.0,2.074074074074074,0.9212121212121213,1.0,1.0
3,DG 34:1,1.1915708812260535,1.7281553398058251,0.8808290155440414,1.2945891783567134,1.4363636363636365,0.6186915887850468,0.9337474120082815,1.068345323741007,1.2979942693409738,...,0.8823529411764706,1.5978835978835977,1.7428571428571429,0.894484412470024,1.728643216080402,0.9384164222873901,3.608540925266904,1.0139470013947,1.064102564102564,1.1683501683501685
4,DG 34:3,1.0,1.0,1.0,1.0,1.0,1.0,1.35,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.5862068965517242,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160,PI 36:2,1.0,0.877887788778878,0.8590225563909775,1.1898989898989898,1.0,0.8542319749216302,0.8272583201267829,1.0,1.0219298245614035,...,1.0,1.0,2.0122850122850124,0.13386880856760375,3.8,4.390000000000001,0.35460992907801414,1.9054878048780488,0.8333333333333333,4.77
161,PI 36:4,1.0,1.0,5.34,0.2557544757033248,1.0,0.21551724137931033,1.0,1.0,0.2159827213822894,...,1.0,1.0,1.0,0.24509803921568626,1.0,1.0,0.4166666666666667,1.0,1.0,1.0
162,PI 38:3,1.0,1.0,1.6099999999999999,0.17699115044247787,1.0,0.4444444444444444,0.4587155963302753,0.7936507936507936,1.2,...,0.6134969325153374,1.0,1.0,0.6329113924050632,1.0,1.0,0.5952380952380952,1.0,1.0,1.0
163,PI 38:4,0.43090734209682585,1.4894249834765365,1.2672684458398744,2.068111455108359,0.42484557309540155,0.6867469879518071,0.5048882681564246,0.42813983599482086,1.108579738245274,...,1.1399253731343284,0.4620563035495716,1.0936768149882905,0.4232223903177004,1.096938775510204,0.9394673123486683,0.9361277445109781,1.4201777013470906,0.5042353990191708,0.4478199718706048


In [45]:
patient_data

,Patient code,Age,Sex,severity
0,BS-064,14,M,severe
1,JV-035,13,M,severe
2,KT-515,29,M,severe
3,KT-926,8,M,severe
4,JV-148,14,M,severe
5,JV-157,13,M,severe
6,KT-193,10,M,severe
7,KT-247,10,F,severe
8,KT-312,14,M,severe
9,KT-347,9,M,severe


In [46]:
# Extraire les informations des patients à partir des colonnes
# Exclure la colonne "Family" et "severity"
sample_cols = sum_species.columns[1:-1]
patient_info = pd.DataFrame({
    # Retirer le jour (D0, D3...) pour matcher patient_data
    "Sample": [col[:6] for col in sample_cols],
    "Day": [col[7:9] for col in sample_cols],
    "Column": sample_cols,
    # Dernière ligne contient la sévérité
    "Severity": sum_species.iloc[-1, 1:-1].values
})

In [47]:
patient_info

,Sample,Day,Column,Severity
0,KT-539,D3,KT-539_D3_D60,mild
1,BS-671,D3,BS-671_D3_D60,mild
2,BS-064,D3,BS-064_D3_D60,severe
3,KT-805,D3,KT-805_D3_D60,mild
4,KT-926,D3,KT-926_D3_D60,severe
5,KT-705,D3,KT-705_D3_D60,severe
6,BS-336,D3,BS-336_D3_D60,mild
7,BS-082,D3,BS-082_D3_D60,mild
8,BS-351,D3,BS-351_D3_D60,mild
9,KT-347,D3,KT-347_D3_D60,severe


In [22]:
# Fusionner avec patient_data pour récupérer le sexe
patient_data = patient_data.rename(columns={"Patient code": "Sample"})
patient_info = patient_info.merge(patient_data, on="Sample", how="left")

In [23]:
patient_info.set_index("Column", inplace=True)
patient_info

,Sample,Day,Severity,Age,Sex,severity
Column,,,,,,
KT-539_D3_D60,KT-539,D3_D60,mild,5,M,mild
BS-671_D3_D60,BS-671,D3_D60,mild,8,F,mild
BS-064_D3_D60,BS-064,D3_D60,severe,14,M,severe
KT-805_D3_D60,KT-805,D3_D60,mild,13,M,mild
KT-926_D3_D60,KT-926,D3_D60,severe,8,M,severe
KT-705_D3_D60,KT-705,D3_D60,severe,17,M,severe
BS-336_D3_D60,BS-336,D3_D60,mild,11,F,mild
BS-082_D3_D60,BS-082,D3_D60,mild,18,M,mild
BS-351_D3_D60,BS-351,D3_D60,mild,7,M,mild


In [24]:
lipid_data = sum_species.iloc[:-1, 1:-1]
lipid_data = lipid_data.astype(float)

In [25]:
# Déterminer un seuil pour chaque famille de lipides (ex : médiane). PS: le log2 a déjà été appliqué
thresholds = lipid_data.median(axis=1)
thresholds

0      1.000000
1      1.173913
2      1.000000
3      1.191571
4      1.000000
         ...   
159    1.000000
160    1.000000
161    1.000000
162    1.000000
163    0.845037
Length: 164, dtype: float64

In [26]:
# Binariser les valeurs des lipides (1 si supérieur au seuil, 0 sinon)
binary_lipid_data = lipid_data.gt(thresholds, axis=0).astype(int)
binary_lipid_data

,KT-539_D3_D60,BS-671_D3_D60,BS-064_D3_D60,KT-805_D3_D60,KT-926_D3_D60,KT-705_D3_D60,BS-336_D3_D60,BS-082_D3_D60,BS-351_D3_D60,KT-347_D3_D60,...,KT-716_D3_D60,KT-974_D3_D60,KT-537_D3_D60,KT-612_D3_D60,KT-445_D3_D60,KT-525_D3_D60,KT-352_D3_D60,JV-148_D3_D60,BS-346_D3_D60,KT-663_D3_D60
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,1,0,1,1,0,0,0,1,1,...,0,0,0,1,0,1,1,1,1,0
2,0,1,1,0,1,0,0,0,1,1,...,1,0,0,1,0,0,0,1,0,0
3,0,1,0,1,1,0,0,0,1,1,...,1,0,1,1,0,1,0,1,0,0
4,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
160,0,0,0,1,0,0,0,0,1,0,...,0,0,0,1,0,1,1,0,1,0
161,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
162,0,0,1,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


## Test du Chi² sur les lipides en fonction de la sévérité

In [27]:
# Effectuer les tests du Chi²
chi2_results = {}

# Chi² entre la sévérité et la présence des familles de lipides
for i, lipid in enumerate(sum_species.iloc[:-1, 0]):
    contingency_table = pd.crosstab(
        binary_lipid_data.iloc[i, :], patient_info["Severity"])
    chi2, p, _, _ = chi2_contingency(contingency_table)
    chi2_results[lipid] = {"Chi2": chi2, "p-value": p}

In [28]:
chi2_results

{'DG 30:0': {'Chi2': 0.0, 'p-value': 1.0},
 'DG 32:0': {'Chi2': 9.242340225563906, 'p-value': 0.002364825341650665},
 'DG 32:1': {'Chi2': 1.8631122448979591, 'p-value': 0.17226603191939072},
 'DG 34:1': {'Chi2': 5.747979323308268, 'p-value': 0.016507634005564046},
 'DG 34:3': {'Chi2': 0.0, 'p-value': 1.0},
 'DG 36:3': {'Chi2': 0.22053571428571378, 'p-value': 0.6386320338113531},
 'DG 34:2': {'Chi2': 3.0795582706766904, 'p-value': 0.07928205591587303},
 'DG 36:1': {'Chi2': 1.2370770676691718, 'p-value': 0.2660351186466848},
 'DG 36:2': {'Chi2': 1.2370770676691718, 'p-value': 0.2660351186466848},
 'DG 36:4': {'Chi2': 1.0532407407407405, 'p-value': 0.3047618940366026},
 'CAR 16:0': {'Chi2': 0.4478064373897709, 'p-value': 0.5033784871225249},
 'CAR 18:1': {'Chi2': 1.2370770676691718, 'p-value': 0.2660351186466848},
 'CAR 18:2': {'Chi2': 0.0, 'p-value': 1.0},
 'CAR 2:0': {'Chi2': 0.0, 'p-value': 1.0},
 'CAR 3:0': {'Chi2': 0.0, 'p-value': 1.0},
 'CAR 8:0': {'Chi2': 0.0, 'p-value': 1.0},
 'CE

De nombreux lipides montrent une association statistiquement significative avec la sévérité de la dengue. Cela suggère que la présence (ou l’absence) de ces lipides seraient liés à la gravité de la maladie. Reste à voir si cette valeur passe la correction de Benjamini-Hochberg.

In [29]:
# Convertir les résultats en DataFrame et sauvegarder
chi2_df = pd.DataFrame.from_dict(chi2_results, orient="index")


In [30]:
chi2_df.to_csv(
    "/Users/loictalignani/research/project/lipidomics/analysis/univariate_analysis/chi2_results_ratio_D03_D60.csv", index=True)

print("Analyse Chi² terminée. Résultats enregistrés dans chi2_results_log2FC_D03.csv")

Analyse Chi² terminée. Résultats enregistrés dans chi2_results_log2FC_D03.csv


## Benjamini-Hochberg correction
Plusieurs tests en parallèle sont effectués, ce qui augmente le risque de faux positifs. La correction BH ajuste les p-values pour tenir compte de ce biais. Elle contrôle le taux de fausses découvertes (FDR) plutôt que le taux d’erreur global. C’est plus adapté que Bonferroni, qui est trop strict. Les p-values corrigées sont souvent plus grandes, ce qui diminue le risque d’interpréter à tort un résultat comme significatif.

In [31]:
# Correction de Benjamini-Hochberg pour toutes les p-values
_, corrected_p_values, _, _ = multipletests(
    chi2_df["p-value"], method="fdr_bh")

# Ajouter les p-values corrigées au DataFrame
chi2_df["p-value (BH corrected)"] = corrected_p_values

# Afficher les résultats
print(chi2_df)

             Chi2   p-value  p-value (BH corrected)
DG 30:0  0.000000  1.000000                1.000000
DG 32:0  9.242340  0.002365                0.193916
DG 32:1  1.863112  0.172266                1.000000
DG 34:1  5.747979  0.016508                0.451209
DG 34:3  0.000000  1.000000                1.000000
...           ...       ...                     ...
PI 34:2  0.000000  1.000000                1.000000
PI 36:2  0.179383  0.671904                1.000000
PI 36:4  1.807705  0.178784                1.000000
PI 38:3  0.000000  1.000000                1.000000
PI 38:4  1.237077  0.266035                1.000000

[164 rows x 3 columns]


In [32]:
chi2_df.to_csv(
    "/Users/loictalignani/research/project/lipidomics/analysis/univariate_analysis/chi2_results_ratios_bh_corrected_D03_D60.csv", index=True)

print("Analyse Chi² terminée. Résultats enregistrés dans chi2_results_ratios_bh_corrected_D03_D60.csv")

Analyse Chi² terminée. Résultats enregistrés dans chi2_results_ratios_bh_corrected_D03_D60.csv


# D10 - données normalisées (ratio D10/D60)

In [48]:
# Charger le fichier Excel
file_path = "/Users/loictalignani/research/project/lipidomics/data/lipidsig_datasets/Ratios/Ratio_lipid_abundance_data_D10_D60.xlsx"
xls = pd.ExcelFile(file_path)

# Charger les données des onglets
sum_species = pd.read_excel(xls, sheet_name="species")
patient_data = pd.read_excel(xls, sheet_name="patient_data")

In [49]:
sum_species

,feature,KT-539_D10_D60,BS-671_D10_D60,BS-064_D10_D60,KT-805_D10_D60,KT-926_D10_D60,KT-705_D10_D60,BS-336_D10_D60,BS-082_D10_D60,BS-351_D10_D60,...,KT-974_D10_D60,KT-537_D10_D60,KT-612_D10_D60,KT-445_D10_D60,KT-525_D10_D60,KT-352_D10_D60,JV-148_D10_D60,BS-346_D10_D60,KT-663_D10_D60,KT-695_D10_D60
0,DG 30:0,1.0,1.0,1.0,1.0,1.0,1.06993006993007,1.0,1.0,1.0,...,1.0,1.0,1.5699999999999998,1.0,1.0,1.5899999999999999,1.58,1.0,1.0,1.0
1,DG 32:0,1.72,0.969879518072289,1.235897435897436,2.005813953488372,1.3669064748201436,0.8811475409836066,1.582191780821918,1.2317880794701985,1.7755102040816328,...,0.6756756756756757,1.44,3.045197740112994,1.639751552795031,2.6100000000000003,3.02,2.1605839416058394,2.5733944954128445,1.9791666666666667,1.92
2,DG 32:1,1.65,1.0,2.73,2.013245033112583,1.0,1.0,1.3924050632911393,1.1891891891891893,2.2199999999999998,...,1.0,1.54,1.3148148148148147,1.370860927152318,1.9,2.84,2.2296296296296294,2.2969696969696973,2.8899999999999997,1.62
3,DG 34:1,2.061302681992337,1.116504854368932,1.0569948186528497,2.346693386773547,1.8472727272727274,0.7570093457943925,1.1780538302277432,1.4999999999999998,1.7392550143266476,...,0.8291316526610645,1.7142857142857142,1.4133333333333333,1.79136690647482,3.391959798994975,2.598240469208211,2.7544483985765127,1.7154811715481173,2.75,1.932659932659933
4,DG 34:3,1.0,1.0,1.0,1.54,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.43,1.1810344827586208,1.73,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160,PI 36:2,5.11,1.155115511551155,1.0281954887218043,0.9818181818181817,1.0,0.829153605015674,0.7797147385103012,1.0,0.7909356725146199,...,1.0,3.51,1.398034398034398,0.7777777777777778,5.04,5.98,0.35460992907801414,1.4725609756097562,2.094202898550724,1.0
161,PI 36:4,10.67,1.0,10.45,0.2557544757033248,1.0,1.2306034482758619,7.87,1.0,1.0907127429805616,...,1.0,1.0,4.76,1.1299019607843135,5.75,1.0,0.4166666666666667,17.55,11.02,6.69
162,PI 38:3,2.02,4.91,9.18,0.39469026548672564,1.0,0.7644444444444445,3.077981651376147,0.7936507936507936,4.45,...,0.9325153374233129,2.59,4.0600000000000005,0.6329113924050632,5.3,5.7,1.5535714285714286,5.75,6.0,1.0
163,PI 38:4,1.3273485091375439,1.2042300066093852,1.9505494505494507,1.8792569659442726,0.9162663006177075,0.8075378436824219,0.986731843575419,0.6348726801899007,0.9854580707707222,...,1.3942786069651745,1.2301101591187271,1.4487119437939109,1.649394856278366,2.8558673469387754,2.2645278450363193,1.5189620758483036,1.29951275437088,2.378065091395453,1.0227848101265824


In [50]:
patient_data

,Patient code,Age,Sex,severity
0,BS-064,14,M,severe
1,JV-035,13,M,severe
2,KT-515,29,M,severe
3,KT-926,8,M,severe
4,JV-148,14,M,severe
5,JV-157,13,M,severe
6,KT-193,10,M,severe
7,KT-247,10,F,severe
8,KT-312,14,M,severe
9,KT-347,9,M,severe


In [53]:
# Extraire les informations des patients à partir des colonnes
# Exclure la colonne "Family" et "severity"
sample_cols = sum_species.columns[1:-1]
patient_info = pd.DataFrame({
    # Retirer le jour (D0, D3...) pour matcher patient_data
    "Sample": [col[:6] for col in sample_cols],
    "Day": [col[7:10] for col in sample_cols],
    "Column": sample_cols,
    # Dernière ligne contient la sévérité
    "Severity": sum_species.iloc[-1, 1:-1].values
})

In [54]:
patient_info

,Sample,Day,Column,Severity
0,KT-539,D10,KT-539_D10_D60,mild
1,BS-671,D10,BS-671_D10_D60,mild
2,BS-064,D10,BS-064_D10_D60,severe
3,KT-805,D10,KT-805_D10_D60,mild
4,KT-926,D10,KT-926_D10_D60,severe
5,KT-705,D10,KT-705_D10_D60,severe
6,BS-336,D10,BS-336_D10_D60,mild
7,BS-082,D10,BS-082_D10_D60,mild
8,BS-351,D10,BS-351_D10_D60,mild
9,KT-347,D10,KT-347_D10_D60,severe


In [55]:
# Fusionner avec patient_data pour récupérer le sexe
patient_data = patient_data.rename(columns={"Patient code": "Sample"})
patient_info = patient_info.merge(patient_data, on="Sample", how="left")

In [56]:
patient_info.set_index("Column", inplace=True)
patient_info

,Sample,Day,Severity,Age,Sex,severity
Column,,,,,,
KT-539_D10_D60,KT-539,D10,mild,5,M,mild
BS-671_D10_D60,BS-671,D10,mild,8,F,mild
BS-064_D10_D60,BS-064,D10,severe,14,M,severe
KT-805_D10_D60,KT-805,D10,mild,13,M,mild
KT-926_D10_D60,KT-926,D10,severe,8,M,severe
KT-705_D10_D60,KT-705,D10,severe,17,M,severe
BS-336_D10_D60,BS-336,D10,mild,11,F,mild
BS-082_D10_D60,BS-082,D10,mild,18,M,mild
BS-351_D10_D60,BS-351,D10,mild,7,M,mild


In [57]:
lipid_data = sum_species.iloc[:-1, 1:-1]
lipid_data = lipid_data.astype(float)

In [58]:
# Déterminer un seuil pour chaque famille de lipides (ex : médiane). PS: le log2 a déjà été appliqué
thresholds = lipid_data.median(axis=1)
thresholds

0      1.000000
1      1.510000
2      1.540000
3      1.691120
4      1.000000
         ...   
159    1.000000
160    1.043178
161    1.573948
162    2.020000
163    1.299513
Length: 164, dtype: float64

In [59]:
# Binariser les valeurs des lipides (1 si supérieur au seuil, 0 sinon)
binary_lipid_data = lipid_data.gt(thresholds, axis=0).astype(int)
binary_lipid_data

,KT-539_D10_D60,BS-671_D10_D60,BS-064_D10_D60,KT-805_D10_D60,KT-926_D10_D60,KT-705_D10_D60,BS-336_D10_D60,BS-082_D10_D60,BS-351_D10_D60,KT-347_D10_D60,...,KT-716_D10_D60,KT-974_D10_D60,KT-537_D10_D60,KT-612_D10_D60,KT-445_D10_D60,KT-525_D10_D60,KT-352_D10_D60,JV-148_D10_D60,BS-346_D10_D60,KT-663_D10_D60
0,0,0,0,0,0,1,0,0,0,0,...,0,0,0,1,0,0,1,1,0,0
1,1,0,0,1,0,0,1,0,1,0,...,0,0,0,1,1,1,1,1,1,1
2,1,0,1,1,0,0,0,0,1,0,...,0,0,0,0,0,1,1,1,1,1
3,1,0,0,1,1,0,0,0,1,0,...,0,0,1,0,1,1,1,1,1,1
4,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,1,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
160,1,1,0,0,0,0,0,0,0,0,...,0,0,1,1,0,1,1,0,1,1
161,1,0,1,0,0,0,1,0,0,1,...,0,0,0,1,0,1,0,0,1,1
162,0,1,1,0,0,0,1,0,1,0,...,0,0,1,1,0,1,1,0,1,1


## Test du Chi² sur les lipides en fonction de la sévérité

In [60]:
# Effectuer les tests du Chi²
chi2_results = {}

# Chi² entre la sévérité et la présence des familles de lipides
for i, lipid in enumerate(sum_species.iloc[:-1, 0]):
    contingency_table = pd.crosstab(
        binary_lipid_data.iloc[i, :], patient_info["Severity"])
    chi2, p, _, _ = chi2_contingency(contingency_table)
    chi2_results[lipid] = {"Chi2": chi2, "p-value": p}

In [61]:
chi2_results

{'DG 30:0': {'Chi2': 1.0532407407407405, 'p-value': 0.3047618940366026},
 'DG 32:0': {'Chi2': 0.0, 'p-value': 1.0},
 'DG 32:1': {'Chi2': 0.22053571428571378, 'p-value': 0.6386320338113531},
 'DG 34:1': {'Chi2': 0.22053571428571378, 'p-value': 0.6386320338113531},
 'DG 34:3': {'Chi2': 0.050781250000000035, 'p-value': 0.8217093743039579},
 'DG 36:3': {'Chi2': 0.0, 'p-value': 1.0},
 'DG 34:2': {'Chi2': 1.2370770676691718, 'p-value': 0.2660351186466848},
 'DG 36:1': {'Chi2': 1.2370770676691718, 'p-value': 0.2660351186466848},
 'DG 36:2': {'Chi2': 0.0, 'p-value': 1.0},
 'DG 36:4': {'Chi2': 0.0, 'p-value': 1.0},
 'CAR 16:0': {'Chi2': 0.0, 'p-value': 1.0},
 'CAR 18:1': {'Chi2': 0.0, 'p-value': 1.0},
 'CAR 18:2': {'Chi2': 0.0, 'p-value': 1.0},
 'CAR 2:0': {'Chi2': 0.0, 'p-value': 1.0},
 'CAR 3:0': {'Chi2': 0.0, 'p-value': 1.0},
 'CAR 8:0': {'Chi2': 0.0, 'p-value': 1.0},
 'CE 16:1': {'Chi2': 0.4234605911330053, 'p-value': 0.5152153080626902},
 'CE 18:1': {'Chi2': 0.0, 'p-value': 1.0},
 'CE 18:2

Aucun lipide ne montre une association statistiquement significative avec la sévérité de la dengue. Cela suggère que la présence (ou l’absence) de ces lipides ne seraient pas liés à la gravité de la maladie.

In [62]:
# Convertir les résultats en DataFrame et sauvegarder
chi2_df = pd.DataFrame.from_dict(chi2_results, orient="index")


In [63]:
chi2_df.to_csv(
    "/Users/loictalignani/research/project/lipidomics/analysis/univariate_analysis/chi2_results_ratios_D10_D60.csv", index=True)

print("Analyse Chi² terminée. Résultats enregistrés dans chi2_results_ratios_D10_D60.csv")

Analyse Chi² terminée. Résultats enregistrés dans chi2_results_ratios_D10_D60.csv


## Benjamini-Hochberg correction
Plusieurs tests en parallèle sont effectués, ce qui augmente le risque de faux positifs. La correction BH ajuste les p-values pour tenir compte de ce biais. Elle contrôle le taux de fausses découvertes (FDR) plutôt que le taux d’erreur global. C’est plus adapté que Bonferroni, qui est trop strict. Les p-values corrigées sont souvent plus grandes, ce qui diminue le risque d’interpréter à tort un résultat comme significatif.

In [64]:
# Correction de Benjamini-Hochberg pour toutes les p-values
_, corrected_p_values, _, _ = multipletests(
    chi2_df["p-value"], method="fdr_bh")

# Ajouter les p-values corrigées au DataFrame
chi2_df["p-value (BH corrected)"] = corrected_p_values

# Afficher les résultats
print(chi2_df)

             Chi2   p-value  p-value (BH corrected)
DG 30:0  1.053241  0.304762                1.000000
DG 32:0  0.000000  1.000000                1.000000
DG 32:1  0.220536  0.638632                1.000000
DG 34:1  0.220536  0.638632                1.000000
DG 34:3  0.050781  0.821709                1.000000
...           ...       ...                     ...
PI 34:2  0.000000  1.000000                1.000000
PI 36:2  0.220536  0.638632                1.000000
PI 36:4  1.237077  0.266035                0.948473
PI 38:3  0.220536  0.638632                1.000000
PI 38:4  3.079558  0.079282                0.948473

[164 rows x 3 columns]


In [65]:
chi2_df.to_csv(
    "/Users/loictalignani/research/project/lipidomics/analysis/univariate_analysis/chi2_results_ratios_bh_corrected_D10_D60.csv", index=True)

print("Analyse Chi² terminée. Résultats enregistrés dans chi2_results_ratios_bh_corrected_D10_D60.csv")

Analyse Chi² terminée. Résultats enregistrés dans chi2_results_ratios_bh_corrected_D10_D60.csv


# D0 vs. D3 - données normalisées (ratio D0/D3)

In [66]:
# Charger le fichier Excel
file_path = "/Users/loictalignani/research/project/lipidomics/data/lipidsig_datasets/Ratios/Ratio_lipid_abundance_data_D0vsD3.xlsx"
xls = pd.ExcelFile(file_path)

# Charger les données des onglets
sum_species = pd.read_excel(xls, sheet_name="species")
patient_data = pd.read_excel(xls, sheet_name="patient_data")

In [67]:
sum_species

,feature,KT-695_D0_D3,KT-723_D0_D3,KT-247_D0_D3,KT-718_D0_D3,KT-880_D0_D3,KT-926_D0_D3,JV-138_D0_D3,KT-974_D0_D3,KT-193_D0_D3,...,BS-377_D0_D3,KT-312_D0_D3,KT-612_D0_D3,JV-035_D0_D3,KT-434_D0_D3,BS-346_D0_D3,KT-705_D0_D3,BS-082_D0_D3,JV-148_D0_D3,KT-347_D0_D3
0,DG 30:0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.42,1.0,1.0,1.0,1.0,1.0,1.0
1,DG 32:0,0.5128205128205129,0.6024096385542168,0.8506787330316742,1.3424657534246576,0.9183673469387755,1.12565445026178,1.1006289308176103,0.7092198581560284,0.8833333333333333,...,0.8090452261306532,0.887323943661972,0.45195729537366547,2.301075268817204,1.0,0.3846153846153846,1.2450331125827814,1.53,0.5076142131979695,1.050314465408805
2,DG 32:1,1.0,0.5050505050505051,1.0,1.0,1.0,1.5693430656934304,1.0,1.0,1.1309523809523807,...,0.4807692307692307,0.5870445344129555,0.5181347150259067,1.7164179104477615,1.0,0.6578947368421053,1.0,1.37,0.35714285714285715,0.7246376811594204
3,DG 34:1,0.648414985590778,0.2848101265822785,0.8133561643835616,0.8466019417475726,1.3449197860962567,2.2860759493670884,0.780373831775701,0.5968253968253968,0.9484240687679082,...,0.5933806146572103,0.5575679172056921,0.22185792349726777,3.2559726962457334,1.209183673469388,0.2558459422283356,1.3957703927492446,1.144781144781145,0.38362919132149903,1.2729468599033813
4,DG 34:3,1.0,0.7246376811594204,1.0,1.0,1.0,1.69,1.0,1.0,1.0,...,1.0,1.0,1.0,1.5699999999999998,1.0,1.0,1.0,1.0,0.5434782608695653,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160,PI 36:2,0.20964360587002098,0.2004008016032064,0.6359300476947536,0.29325513196480935,1.0,3.2,0.3184713375796178,1.0,1.109014675052411,...,1.5623582766439907,1.1127450980392155,0.12210012210012208,1.0744336569579287,3.17,0.08,0.7706422018348624,1.0,4.609999999999999,0.15673981191222572
161,PI 36:4,1.0,1.0,1.0,1.0,1.0,4.63,1.0,1.0,1.0,...,1.0,0.1838235294117647,1.0,0.14492753623188406,1.0,1.0,1.0,1.0,1.0,1.0
162,PI 38:3,1.0,1.0,0.8695652173913044,1.0,1.0,1.52,1.0,1.0,1.0,...,0.45454545454545453,0.6192893401015228,1.0,3.906593406593407,1.0,1.0,1.0,1.0,1.0,1.0
163,PI 38:4,0.3021356783919598,0.5101495726495727,0.41036456764085566,0.49704877291084193,1.7688073394495414,1.245557350565428,0.7353286384976526,0.20676486633933444,1.004885993485342,...,0.975229724330803,0.5493053202304303,0.21456102783725908,0.3064098878988215,1.918793503480278,0.18748738647830474,0.586144849302744,0.688508064516129,0.8272921108742003,0.44347063978965817


In [68]:
patient_data

,Patient code,Age,Sex,severity
0,BS-064,14,M,severe
1,JV-035,13,M,severe
2,KT-515,29,M,severe
3,KT-926,8,M,severe
4,JV-148,14,M,severe
5,JV-157,13,M,severe
6,KT-193,10,M,severe
7,KT-247,10,F,severe
8,KT-312,14,M,severe
9,KT-347,9,M,severe


In [71]:
# Extraire les informations des patients à partir des colonnes
# Exclure la colonne "Family" et "severity"
sample_cols = sum_species.columns[1:-1]
patient_info = pd.DataFrame({
    # Retirer le jour (D0, D3...) pour matcher patient_data
    "Sample": [col[:6] for col in sample_cols],
    "Day": [col[7:9] for col in sample_cols],
    "Column": sample_cols,
    # Dernière ligne contient la sévérité
    "Severity": sum_species.iloc[-1, 1:-1].values
})

In [72]:
patient_info

,Sample,Day,Column,Severity
0,KT-695,D0,KT-695_D0_D3,mild
1,KT-723,D0,KT-723_D0_D3,mild
2,KT-247,D0,KT-247_D0_D3,severe
3,KT-718,D0,KT-718_D0_D3,severe
4,KT-880,D0,KT-880_D0_D3,mild
5,KT-926,D0,KT-926_D0_D3,severe
6,JV-138,D0,JV-138_D0_D3,mild
7,KT-974,D0,KT-974_D0_D3,mild
8,KT-193,D0,KT-193_D0_D3,severe
9,KT-412,D0,KT-412_D0_D3,mild


In [73]:
# Fusionner avec patient_data pour récupérer le sexe
patient_data = patient_data.rename(columns={"Patient code": "Sample"})
patient_info = patient_info.merge(patient_data, on="Sample", how="left")

In [74]:
patient_info.set_index("Column", inplace=True)
patient_info

,Sample,Day,Severity,Age,Sex,severity
Column,,,,,,
KT-695_D0_D3,KT-695,D0,mild,12,M,mild
KT-723_D0_D3,KT-723,D0,mild,8,M,mild
KT-247_D0_D3,KT-247,D0,severe,10,F,severe
KT-718_D0_D3,KT-718,D0,severe,15,F,severe
KT-880_D0_D3,KT-880,D0,mild,19,F,mild
KT-926_D0_D3,KT-926,D0,severe,8,M,severe
JV-138_D0_D3,JV-138,D0,mild,13,M,mild
KT-974_D0_D3,KT-974,D0,mild,14,M,mild
KT-193_D0_D3,KT-193,D0,severe,10,M,severe


In [75]:
lipid_data = sum_species.iloc[:-1, 1:-1]
lipid_data = lipid_data.astype(float)

In [76]:
# Déterminer un seuil pour chaque famille de lipides (ex : médiane). PS: le log2 a déjà été appliqué
thresholds = lipid_data.median(axis=1)
thresholds

0      1.000000
1      0.871429
2      1.000000
3      0.641176
4      1.000000
         ...   
159    1.000000
160    0.838017
161    1.000000
162    1.000000
163    0.586145
Length: 164, dtype: float64

In [77]:
# Binariser les valeurs des lipides (1 si supérieur au seuil, 0 sinon)
binary_lipid_data = lipid_data.gt(thresholds, axis=0).astype(int)
binary_lipid_data

,KT-695_D0_D3,KT-723_D0_D3,KT-247_D0_D3,KT-718_D0_D3,KT-880_D0_D3,KT-926_D0_D3,JV-138_D0_D3,KT-974_D0_D3,KT-193_D0_D3,KT-412_D0_D3,...,KT-445_D0_D3,BS-377_D0_D3,KT-312_D0_D3,KT-612_D0_D3,JV-035_D0_D3,KT-434_D0_D3,BS-346_D0_D3,KT-705_D0_D3,BS-082_D0_D3,JV-148_D0_D3
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
1,0,0,0,1,1,1,1,0,1,0,...,0,0,1,0,1,1,0,1,1,0
2,0,0,0,0,0,1,0,0,1,0,...,0,0,0,0,1,0,0,0,1,0
3,1,0,1,1,1,1,1,0,1,0,...,0,0,0,0,1,1,0,1,1,0
4,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
160,0,0,0,0,1,1,0,1,1,0,...,1,1,1,0,1,1,0,0,1,1
161,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
162,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0


## Test du Chi² sur les lipides en fonction de la sévérité

In [78]:
# Effectuer les tests du Chi²
chi2_results = {}

# Chi² entre la sévérité et la présence des familles de lipides
for i, lipid in enumerate(sum_species.iloc[:-1, 0]):
    contingency_table = pd.crosstab(
        binary_lipid_data.iloc[i, :], patient_info["Severity"])
    chi2, p, _, _ = chi2_contingency(contingency_table)
    chi2_results[lipid] = {"Chi2": chi2, "p-value": p}

In [79]:
chi2_results

{'DG 30:0': {'Chi2': 0.017150999155643137, 'p-value': 0.8958055078502888},
 'DG 32:0': {'Chi2': 2.05324901491697, 'p-value': 0.15188176923500454},
 'DG 32:1': {'Chi2': 0.14256768048128354, 'p-value': 0.7057416168308079},
 'DG 34:1': {'Chi2': 0.6191510695187157, 'p-value': 0.4313629288311962},
 'DG 34:3': {'Chi2': 0.8458501951149009, 'p-value': 0.3577289544171932},
 'DG 36:3': {'Chi2': 2.05324901491697, 'p-value': 0.15188176923500454},
 'DG 34:2': {'Chi2': 0.6191510695187157, 'p-value': 0.4313629288311962},
 'DG 36:1': {'Chi2': 0.0, 'p-value': 1.0},
 'DG 36:2': {'Chi2': 0.6191510695187157, 'p-value': 0.4313629288311962},
 'DG 36:4': {'Chi2': 0.0, 'p-value': 1.0},
 'CAR 16:0': {'Chi2': 1.325497818744724, 'p-value': 0.24960774253978762},
 'CAR 18:1': {'Chi2': 1.325497818744724, 'p-value': 0.24960774253978762},
 'CAR 18:2': {'Chi2': 3.2304935969603172, 'p-value': 0.07227890025823178},
 'CAR 2:0': {'Chi2': 0.0, 'p-value': 1.0},
 'CAR 3:0': {'Chi2': 0.8458501951149009, 'p-value': 0.357728954

Quelques lipides montrent une association statistiquement significative avec la sévérité de la dengue.

In [80]:
# Convertir les résultats en DataFrame et sauvegarder
chi2_df = pd.DataFrame.from_dict(chi2_results, orient="index")


In [81]:
chi2_df.to_csv(
    "/Users/loictalignani/research/project/lipidomics/analysis/univariate_analysis/chi2_results_ratios_D0_D3.csv", index=True)

print("Analyse Chi² terminée. Résultats enregistrés dans chi2_results_ratios_D0_D3.csv")

Analyse Chi² terminée. Résultats enregistrés dans chi2_results_ratios_D0_D3.csv


## Benjamini-Hochberg correction
Plusieurs tests en parallèle sont effectués, ce qui augmente le risque de faux positifs. La correction BH ajuste les p-values pour tenir compte de ce biais. Elle contrôle le taux de fausses découvertes (FDR) plutôt que le taux d’erreur global. C’est plus adapté que Bonferroni, qui est trop strict. Les p-values corrigées sont souvent plus grandes, ce qui diminue le risque d’interpréter à tort un résultat comme significatif.

In [82]:
# Correction de Benjamini-Hochberg pour toutes les p-values
_, corrected_p_values, _, _ = multipletests(
    chi2_df["p-value"], method="fdr_bh")

# Ajouter les p-values corrigées au DataFrame
chi2_df["p-value (BH corrected)"] = corrected_p_values

# Afficher les résultats
print(chi2_df)

             Chi2   p-value  p-value (BH corrected)
DG 30:0  0.017151  0.895806                1.000000
DG 32:0  2.053249  0.151882                0.655490
DG 32:1  0.142568  0.705742                1.000000
DG 34:1  0.619151  0.431363                0.969089
DG 34:3  0.845850  0.357729                0.969089
...           ...       ...                     ...
PI 34:2  0.000000  1.000000                1.000000
PI 36:2  0.619151  0.431363                0.969089
PI 36:4  0.017151  0.895806                1.000000
PI 38:3  0.648200  0.420757                0.969089
PI 38:4  0.000000  1.000000                1.000000

[164 rows x 3 columns]


In [83]:
chi2_df.to_csv(
    "/Users/loictalignani/research/project/lipidomics/analysis/univariate_analysis/chi2_results_ratios_bh_corrected_D0_D3.csv", index=True)

print("Analyse Chi² terminée. Résultats enregistrés dans chi2_results_ratios_bh_corrected_D0_D3.csv")

Analyse Chi² terminée. Résultats enregistrés dans chi2_results_ratios_bh_corrected_D0_D3.csv


# D3 vs. D10 - données normalisées (ratio D3/D10)

In [84]:
# Charger le fichier Excel
file_path = "/Users/loictalignani/research/project/lipidomics/data/lipidsig_datasets/Ratios/Ratio_lipid_abundance_data_D3vsD10.xlsx"
xls = pd.ExcelFile(file_path)

# Charger les données des onglets
sum_species = pd.read_excel(xls, sheet_name="species")
patient_data = pd.read_excel(xls, sheet_name="patient_data")

In [85]:
sum_species

,feature,KT-695_D3_D10,KT-723_D3_D10,KT-247_D3_D10,KT-718_D3_D10,KT-880_D3_D10,KT-926_D3_D10,JV-138_D3_D10,KT-974_D3_D10,KT-193_D3_D10,...,BS-377_D3_D10,KT-312_D3_D10,KT-612_D3_D10,JV-035_D3_D10,KT-434_D3_D10,BS-346_D3_D10,KT-705_D3_D10,BS-082_D3_D10,JV-148_D3_D10,KT-347_D3_D10
0,DG 30:0,1.0,1.0,1.0,1.0,1.0,1.0,0.628930817610063,1.0,1.0,...,0.5102040816326531,0.46511627906976744,0.6369426751592357,0.4444444444444444,1.0,1.0,0.6535947712418301,1.0,0.6329113924050632,1.0
1,DG 32:0,1.015625,0.6916666666666668,0.9285714285714286,0.7978142076502732,0.6735395189003436,1.0052631578947369,0.2939001848428835,1.41,1.2552301255230127,...,0.5887573964497042,0.3408,0.5213358070500929,0.49076517150395776,0.6535947712418301,0.46345811051693403,0.7023255813953488,0.5376344086021506,0.6655405405405406,1.052980132450331
2,DG 32:1,0.6172839506172839,1.0153846153846153,0.4608294930875576,0.5617977528089888,0.37453183520599254,1.37,0.2695417789757412,1.0,0.8195121951219514,...,0.6797385620915033,0.6551724137931034,0.9061032863849766,0.3941176470588235,0.641025641025641,0.40105540897097625,0.43103448275862066,0.5681818181818182,0.9302325581395349,0.9387755102040816
3,DG 34:1,0.6045296167247387,0.9828926905132194,0.6862514688601645,0.7113259668508287,0.344066237350506,0.7775590551181103,0.41074856046065256,1.0641891891891893,2.056974459724951,...,1.0972762645914398,0.9961340206185568,1.2331536388140163,0.5961342828077315,0.42332613390928725,0.5910569105691056,0.817283950617284,0.712230215827338,1.310077519379845,0.9430523917995445
4,DG 34:3,1.0,1.38,1.0,1.0,1.0,1.0,0.6451612903225806,1.0,1.0,...,1.0,0.6711409395973155,1.0,1.0,1.0,0.5780346820809249,1.0,1.0,1.3430656934306566,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160,PI 36:2,4.77,1.401685393258427,1.0448504983388704,0.39467592592592593,0.0992063492063492,1.0,0.48085758039816234,1.0,0.6625,...,0.8732673267326734,0.7338129496402878,1.4393673110720564,1.0729166666666667,0.30303030303030304,1.2939958592132506,1.0302457466918715,1.0,1.0,6.38
161,PI 36:4,0.14947683109118085,1.0,0.1972386587771203,0.21231422505307856,0.048285852245292124,1.0,0.078003120124805,1.0,0.08620689655172414,...,0.19342359767891684,0.5614035087719299,0.21008403361344538,0.859277708592777,1.0,0.05698005698005698,0.17513134851138354,1.0,1.0,0.2506265664160401
162,PI 38:3,1.0,1.0,0.34848484848484845,0.2785515320334262,0.1724137931034483,1.0,0.17667844522968199,0.6578947368421053,0.11013215859030837,...,0.4408817635270541,0.3900990099009901,0.2463054187192118,0.31487889273356395,0.5524861878453039,0.17391304347826086,0.5813953488372093,1.0,0.38314176245210724,1.0
163,PI 38:4,0.43784378437843785,0.801713062098501,0.7373916907353921,0.671184320266889,0.09433962264150943,0.4636704119850188,0.4062947067238913,0.8175735950044601,0.4745893719806763,...,0.6810884353741496,1.0684286748732803,0.7549304881991594,1.046630565583634,0.2310991957104558,1.0928539920599911,0.8504208110175976,0.6743711760707002,0.6162943495400789,0.781239301609038


In [86]:
patient_data

,Patient code,Age,Sex,severity
0,BS-064,14,M,severe
1,JV-035,13,M,severe
2,KT-515,29,M,severe
3,KT-926,8,M,severe
4,JV-148,14,M,severe
5,JV-157,13,M,severe
6,KT-193,10,M,severe
7,KT-247,10,F,severe
8,KT-312,14,M,severe
9,KT-347,9,M,severe


In [87]:
# Extraire les informations des patients à partir des colonnes
# Exclure la colonne "Family" et "severity"
sample_cols = sum_species.columns[1:-1]
patient_info = pd.DataFrame({
    # Retirer le jour (D0, D3...) pour matcher patient_data
    "Sample": [col[:6] for col in sample_cols],
    "Day": [col[7:9] for col in sample_cols],
    "Column": sample_cols,
    # Dernière ligne contient la sévérité
    "Severity": sum_species.iloc[-1, 1:-1].values
})

In [88]:
patient_info

,Sample,Day,Column,Severity
0,KT-695,D3,KT-695_D3_D10,mild
1,KT-723,D3,KT-723_D3_D10,mild
2,KT-247,D3,KT-247_D3_D10,severe
3,KT-718,D3,KT-718_D3_D10,severe
4,KT-880,D3,KT-880_D3_D10,mild
5,KT-926,D3,KT-926_D3_D10,severe
6,JV-138,D3,JV-138_D3_D10,mild
7,KT-974,D3,KT-974_D3_D10,mild
8,KT-193,D3,KT-193_D3_D10,severe
9,KT-412,D3,KT-412_D3_D10,mild


In [89]:
# Fusionner avec patient_data pour récupérer le sexe
patient_data = patient_data.rename(columns={"Patient code": "Sample"})
patient_info = patient_info.merge(patient_data, on="Sample", how="left")

In [90]:
patient_info.set_index("Column", inplace=True)
patient_info

,Sample,Day,Severity,Age,Sex,severity
Column,,,,,,
KT-695_D3_D10,KT-695,D3,mild,12,M,mild
KT-723_D3_D10,KT-723,D3,mild,8,M,mild
KT-247_D3_D10,KT-247,D3,severe,10,F,severe
KT-718_D3_D10,KT-718,D3,severe,15,F,severe
KT-880_D3_D10,KT-880,D3,mild,19,F,mild
KT-926_D3_D10,KT-926,D3,severe,8,M,severe
JV-138_D3_D10,JV-138,D3,mild,13,M,mild
KT-974_D3_D10,KT-974,D3,mild,14,M,mild
KT-193_D3_D10,KT-193,D3,severe,10,M,severe


In [91]:
lipid_data = sum_species.iloc[:-1, 1:-1]
lipid_data = lipid_data.astype(float)

In [92]:
# Déterminer un seuil pour chaque famille de lipides (ex : médiane). PS: le log2 a déjà été appliqué
thresholds = lipid_data.median(axis=1)
thresholds

0      1.000000
1      0.673540
2      0.647343
3      0.777559
4      1.000000
         ...   
159    1.000000
160    1.000000
161    0.198020
162    0.357143
163    0.671184
Length: 164, dtype: float64

In [93]:
# Binariser les valeurs des lipides (1 si supérieur au seuil, 0 sinon)
binary_lipid_data = lipid_data.gt(thresholds, axis=0).astype(int)
binary_lipid_data

,KT-695_D3_D10,KT-723_D3_D10,KT-247_D3_D10,KT-718_D3_D10,KT-880_D3_D10,KT-926_D3_D10,JV-138_D3_D10,KT-974_D3_D10,KT-193_D3_D10,KT-412_D3_D10,...,KT-445_D3_D10,BS-377_D3_D10,KT-312_D3_D10,KT-612_D3_D10,JV-035_D3_D10,KT-434_D3_D10,BS-346_D3_D10,KT-705_D3_D10,BS-082_D3_D10,JV-148_D3_D10
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,1,1,1,0,1,0,1,1,1,...,0,0,0,0,0,0,0,1,0,0
2,0,1,0,0,0,1,0,1,1,1,...,0,1,1,1,0,0,0,0,0,1
3,0,1,0,0,0,0,0,1,1,1,...,0,1,1,1,0,0,0,1,0,1
4,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
160,1,1,1,0,0,0,0,0,0,0,...,0,0,0,1,1,0,1,1,0,0
161,0,1,0,1,0,1,0,1,0,1,...,1,0,1,1,1,1,0,0,1,1
162,1,1,0,0,0,1,0,1,0,1,...,1,1,1,0,0,1,0,1,1,1


## Test du Chi² sur les lipides en fonction de la sévérité

In [94]:
# Effectuer les tests du Chi²
chi2_results = {}

# Chi² entre la sévérité et la présence des familles de lipides
for i, lipid in enumerate(sum_species.iloc[:-1, 0]):
    contingency_table = pd.crosstab(
        binary_lipid_data.iloc[i, :], patient_info["Severity"])
    chi2, p, _, _ = chi2_contingency(contingency_table)
    chi2_results[lipid] = {"Chi2": chi2, "p-value": p}

In [95]:
chi2_results

{'DG 30:0': {'Chi2': 0.0, 'p-value': 1.0},
 'DG 32:0': {'Chi2': 0.019826555023923303, 'p-value': 0.8880225706938752},
 'DG 32:1': {'Chi2': 0.6191510695187157, 'p-value': 0.4313629288311962},
 'DG 34:1': {'Chi2': 0.6191510695187157, 'p-value': 0.4313629288311962},
 'DG 34:3': {'Chi2': 0.0, 'p-value': 1.0},
 'DG 36:3': {'Chi2': 0.019826555023923303, 'p-value': 0.8880225706938752},
 'DG 34:2': {'Chi2': 0.6191510695187157, 'p-value': 0.4313629288311962},
 'DG 36:1': {'Chi2': 0.6191510695187157, 'p-value': 0.4313629288311962},
 'DG 36:2': {'Chi2': 0.019826555023923303, 'p-value': 0.8880225706938752},
 'DG 36:4': {'Chi2': 0.6560478480248406, 'p-value': 0.4179586786429723},
 'CAR 16:0': {'Chi2': 0.8849484339190227, 'p-value': 0.34684991572714086},
 'CAR 18:1': {'Chi2': 2.05324901491697, 'p-value': 0.15188176923500454},
 'CAR 18:2': {'Chi2': 0.0, 'p-value': 1.0},
 'CAR 2:0': {'Chi2': 0.0, 'p-value': 1.0},
 'CAR 3:0': {'Chi2': 0.0, 'p-value': 1.0},
 'CAR 8:0': {'Chi2': 0.0, 'p-value': 1.0},
 'C

Quelques lipides montrent une association statistiquement significative avec la sévérité de la dengue.

In [96]:
# Convertir les résultats en DataFrame et sauvegarder
chi2_df = pd.DataFrame.from_dict(chi2_results, orient="index")


In [97]:
chi2_df.to_csv(
    "/Users/loictalignani/research/project/lipidomics/analysis/univariate_analysis/chi2_results_ratios_D3_D10.csv", index=True)

print("Analyse Chi² terminée. Résultats enregistrés dans chi2_results_ratios_D0_D3.csv")

Analyse Chi² terminée. Résultats enregistrés dans chi2_results_ratios_D0_D3.csv


## Benjamini-Hochberg correction
Plusieurs tests en parallèle sont effectués, ce qui augmente le risque de faux positifs. La correction BH ajuste les p-values pour tenir compte de ce biais. Elle contrôle le taux de fausses découvertes (FDR) plutôt que le taux d’erreur global. C’est plus adapté que Bonferroni, qui est trop strict. Les p-values corrigées sont souvent plus grandes, ce qui diminue le risque d’interpréter à tort un résultat comme significatif.

In [98]:
# Correction de Benjamini-Hochberg pour toutes les p-values
_, corrected_p_values, _, _ = multipletests(
    chi2_df["p-value"], method="fdr_bh")

# Ajouter les p-values corrigées au DataFrame
chi2_df["p-value (BH corrected)"] = corrected_p_values

# Afficher les résultats
print(chi2_df)

             Chi2   p-value  p-value (BH corrected)
DG 30:0  0.000000  1.000000                     1.0
DG 32:0  0.019827  0.888023                     1.0
DG 32:1  0.619151  0.431363                     1.0
DG 34:1  0.619151  0.431363                     1.0
DG 34:3  0.000000  1.000000                     1.0
...           ...       ...                     ...
PI 34:2  0.000000  1.000000                     1.0
PI 36:2  0.000000  1.000000                     1.0
PI 36:4  0.019827  0.888023                     1.0
PI 38:3  0.255275  0.613385                     1.0
PI 38:4  0.000000  1.000000                     1.0

[164 rows x 3 columns]


In [99]:
chi2_df.to_csv(
    "/Users/loictalignani/research/project/lipidomics/analysis/univariate_analysis/chi2_results_ratios_bh_corrected_D3_D10.csv", index=True)

print("Analyse Chi² terminée. Résultats enregistrés dans chi2_results_ratios_bh_corrected_D3_D10.csv")

Analyse Chi² terminée. Résultats enregistrés dans chi2_results_ratios_bh_corrected_D3_D10.csv
